# Phase 2 Integration Proof of Concept

This notebook creates the common 90 m Highline patch grid and verifies that all six Phase 2 data sources can be summarized onto the same modeling unit.

In [1]:
import ee
import pandas as pd
from pathlib import Path

ee.Initialize(project="ee-wildfire-recovery-vasant")

HIGHLINE_EVENT_ID = "ID4549011522720170729"

mtbs_boundaries = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

highline = ee.Feature(
    mtbs_boundaries
    .filter(ee.Filter.eq("Event_ID", HIGHLINE_EVENT_ID))
    .first()
)

highline_geom = highline.geometry()

print("Fire:", highline.get("Incid_Name").getInfo())
print("Event ID:", highline.get("Event_ID").getInfo())
print("Area (km²):", round(highline_geom.area().divide(1e6).getInfo(), 2))

Fire: HIGHLINE
Event ID: ID4549011522720170729
Area (km²): 378.31


## Step 1 — Create the common 90 m project grid

### Step 1A — Define grid projection and alignment

The integration POC uses a fixed 90 m grid in UTM Zone 11N (`EPSG:32611`) for Highline.

This meter-based projection allows each modeling unit to represent a consistent 90 m × 90 m ground area. All source datasets will later be summarized geographically onto these fixed patches rather than assumed to share the same native raster grid.

In [2]:
GRID_CRS = "EPSG:32611"
GRID_SCALE = 90

grid_projection = ee.Projection(GRID_CRS).atScale(GRID_SCALE)

print("Grid CRS:", grid_projection.crs().getInfo())
print("Grid nominal scale (m):", grid_projection.nominalScale().getInfo())
print("Grid transform:", grid_projection.transform().getInfo())

Grid CRS: EPSG:32611
Grid nominal scale (m): 90
Grid transform: PARAM_MT["Affine", 
  PARAMETER["num_row", 3], 
  PARAMETER["num_col", 3], 
  PARAMETER["elt_0_0", 90.0], 
  PARAMETER["elt_1_1", 90.0]]


#### Step 1A observations

Highline is represented by MTBS as an irregular vector fire-perimeter polygon, not as a 30 m raster.

The project defines a separate master modeling grid in UTM Zone 11N (`EPSG:32611`) with fixed 90 m × 90 m cells. Source rasters retain their native grids and will later be summarized geographically onto these common patches.

### Step 1B — Generate the 90 m bounding-box grid

A fixed 90 m grid aligned to the project projection is generated across the rectangular bounding box surrounding Highline.

Using the simple bounding box avoids expensive vector operations against the detailed fire perimeter. The resulting cells remain fixed 90 m × 90 m squares; cells belonging to the Highline study area are selected separately in the next step.

In [3]:
highline_bounds = highline_geom.bounds(
    maxError=1,
    proj=grid_projection
)

highline_grid_bbox = highline_bounds.coveringGrid(
    proj=grid_projection
)

grid_count_bbox = highline_grid_bbox.size().getInfo()

first_cell = ee.Feature(highline_grid_bbox.first())

print("90 m cells in Highline bounding box:", grid_count_bbox)

print(
    "Example cell ground area (m²):",
    round(first_cell.geometry().area(maxError=1).getInfo(), 2)
)
print(
    "Example grid index:",
    first_cell.get("system:index").getInfo()
)

90 m cells in Highline bounding box: 92732
Example cell ground area (m²): 8084.51
Example grid index: 7040,55639


#### Step 1B observations

The Highline bounding box contains 92,732 candidate 90 m grid cells.

Each cell represents approximately 90 m × 90 m in the UTM projection. The small difference between the nominal 8,100 m² projected area and the calculated ground area reflects projection/geodesic effects rather than a grid-size problem.

### Step 1C — Select Highline patches using fire-perimeter overlap

Each fixed 90 m grid cell is evaluated by the fraction of its area lying within the Highline fire perimeter.

For the POC, cells with at least 90% overlap are retained as candidate modeling patches. This preserves fixed 90 m × 90 m units while avoiding patches that lie substantially outside the focal-fire perimeter.

The overlap threshold is provisional and can be audited during production scaling. Reported represented area is based on the nominal area of retained grid cells and is used as a coverage diagnostic rather than an exact geometric measurement.

In [4]:
FIRE_OVERLAP_THRESHOLD = 0.90

# Raster representing area inside the Highline perimeter.
fire_area = (
    ee.Image.pixelArea()
    .clip(highline_geom)
    .rename("fire_area")
)

# Calculate how much Highline area falls inside each candidate 90 m cell.
grid_with_fire_area = fire_area.reduceRegions(
    collection=highline_grid_bbox,
    reducer=ee.Reducer.sum(),
    scale=30,
    crs=GRID_CRS,
    tileScale=4,
)

def add_overlap_and_id(feature):
    fire_area_m2 = ee.Number(
        ee.Algorithms.If(
            feature.get("sum"),
            feature.get("sum"),
            0
        )
    )

    patch_area_m2 = GRID_SCALE * GRID_SCALE

    overlap_fraction = (
        fire_area_m2
        .divide(patch_area_m2)
        .max(0)
        .min(1)
    )

    grid_index = ee.String(feature.get("system:index"))

    patch_id = (
        ee.String("HIGHLINE_")
        .cat(grid_index.replace(",", "_"))
    )

    return feature.set({
        "fire_overlap_fraction": overlap_fraction,
        "patch_id": patch_id,
    })


grid_with_overlap = grid_with_fire_area.map(add_overlap_and_id)

highline_patches = grid_with_overlap.filter(
    ee.Filter.gte(
        "fire_overlap_fraction",
        FIRE_OVERLAP_THRESHOLD
    )
)

patch_count = highline_patches.size().getInfo()

example_patch = ee.Feature(highline_patches.first())

highline_area_km2 = highline_geom.area().divide(1e6).getInfo()

represented_area_km2 = patch_count * (GRID_SCALE * GRID_SCALE) / 1e6

represented_pct = 100 * represented_area_km2 / highline_area_km2

print("Retained Highline patches:", patch_count)

print(
    "Example fire-overlap fraction:",
    round(example_patch.get("fire_overlap_fraction").getInfo(), 4)
)

print(
    "Example patch ID:",
    example_patch.get("patch_id").getInfo()
)

print(
    "Approximate represented area (km²):",
    round(represented_area_km2, 2)
)

print(
    "Approximate Highline area represented (%):",
    round(represented_pct, 2)
)

Retained Highline patches: 45505
Example fire-overlap fraction: 1
Example patch ID: HIGHLINE_7212_55640
Approximate represented area (km²): 368.59
Approximate Highline area represented (%): 97.43


#### Step 1C observations

Using a minimum 90% fire-perimeter overlap threshold retained 45,505 Highline patches.

- Nominal represented area: 368.59 km²
- Approximate fraction of Highline area represented: 97.43%

The retained grid therefore preserves nearly the entire fire footprint while removing most boundary cells that fall substantially outside the perimeter. Each retained patch remains a fixed 90 m × 90 m modeling unit with a deterministic `patch_id`.

## Step 2 — Attach fire, vegetation, and terrain context

The fixed 90 m Highline patches are used as the common spatial unit for integrating MTBS, LANDFIRE, and 3DEP.

Each source retains its native raster grid and is summarized geographically onto the existing project patches. This adds focal-fire, pre-fire vegetation, and terrain information without changing the 90 m modeling units.

### Step 2A — Summarize MTBS focal-fire information

MTBS focal-fire metadata and the 2017 burn-severity raster are attached to the fixed 90 m Highline patches.

For each patch, the geographic area assigned to each MTBS severity class is divided by the nominal 8,100 m² patch area. This produces severity fractions while preserving partial overlap between native MTBS pixels and the project grid.

The focal-fire ID, year, ignition month, and mapped fire size are also retained. Prior-fire and reburn QA are handled separately later.

In [27]:
# Load Highline MTBS metadata and severity
FIRE_YEAR = 2017
PATCH_AREA_M2 = GRID_SCALE * GRID_SCALE

# Focal-fire metadata already loaded in Step 1 as `highline`.
fire_date = ee.Date(highline.get("Ig_Date"))

fire_name = highline.get("Incid_Name")
fire_size_acres = highline.get("BurnBndAc")
ignition_month = fire_date.get("month")

# Load the exact 2017 CONUS MTBS severity mosaic.
mtbs_severity = ee.ImageCollection(
    "USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1"
)

severity_2017 = ee.Image(
    mtbs_severity
    .filter(ee.Filter.eq("system:index", "mtbs_CONUS_2017"))
    .first()
).select("Severity")

print("Fire:", fire_name.getInfo())
print("Fire year:", FIRE_YEAR)
print("Ignition month:", ignition_month.getInfo())
print("Fire size (acres):", fire_size_acres.getInfo())
print("Severity bands:", severity_2017.bandNames().getInfo())
print(
    "Severity nominal scale (m):",
    severity_2017.projection().nominalScale().getInfo()
)

Fire: HIGHLINE
Fire year: 2017
Ignition month: 7
Fire size (acres): 93699
Severity bands: ['Severity']
Severity nominal scale (m): 30


In [28]:
# Calculate severity area inside every 90 m patch
# Geographic pixel area in square meters.
pixel_area = ee.Image.pixelArea()

# MTBS classes 1–5 contain mapped burn-severity information.
mtbs_valid_mask = (
    severity_2017.gte(1)
    .And(severity_2017.lte(5))
)

mtbs_area_bands = ee.Image.cat([
    pixel_area
        .multiply(mtbs_valid_mask)
        .updateMask(severity_2017.mask())
        .rename("mtbs_valid_area_m2"),

    pixel_area
        .multiply(severity_2017.eq(1))
        .updateMask(severity_2017.mask())
        .rename("mtbs_unburned_low_area_m2"),

    pixel_area
        .multiply(severity_2017.eq(2))
        .updateMask(severity_2017.mask())
        .rename("mtbs_low_area_m2"),

    pixel_area
        .multiply(severity_2017.eq(3))
        .updateMask(severity_2017.mask())
        .rename("mtbs_moderate_area_m2"),

    pixel_area
        .multiply(severity_2017.eq(4))
        .updateMask(severity_2017.mask())
        .rename("mtbs_high_area_m2"),

    pixel_area
        .multiply(severity_2017.eq(5))
        .updateMask(severity_2017.mask())
        .rename("mtbs_increased_greenness_area_m2"),

    pixel_area
        .multiply(severity_2017.eq(6))
        .updateMask(severity_2017.mask())
        .rename("mtbs_nonmapping_area_m2"),
])

patches_mtbs_raw = mtbs_area_bands.reduceRegions(
    collection=highline_patches,
    reducer=ee.Reducer.sum(),
    scale=30,
    crs=severity_2017.projection(),
    tileScale=4,
)

In [29]:
# Convert areas to patch fractions and attach fire metadata
def safe_number(feature, field):
    return ee.Number(
        ee.Algorithms.If(
            feature.get(field),
            feature.get(field),
            0
        )
    )


def add_mtbs_summary(feature):
    valid = safe_number(
        feature, "mtbs_valid_area_m2"
    ).divide(PATCH_AREA_M2)

    unburned_low = safe_number(
        feature, "mtbs_unburned_low_area_m2"
    ).divide(PATCH_AREA_M2)

    low = safe_number(
        feature, "mtbs_low_area_m2"
    ).divide(PATCH_AREA_M2)

    moderate = safe_number(
        feature, "mtbs_moderate_area_m2"
    ).divide(PATCH_AREA_M2)

    high = safe_number(
        feature, "mtbs_high_area_m2"
    ).divide(PATCH_AREA_M2)

    increased = safe_number(
        feature, "mtbs_increased_greenness_area_m2"
    ).divide(PATCH_AREA_M2)

    nonmapping = safe_number(
        feature, "mtbs_nonmapping_area_m2"
    ).divide(PATCH_AREA_M2)

    meaningful_burn = low.add(moderate).add(high)

    return feature.set({
        "fire_id": HIGHLINE_EVENT_ID,
        "fire_name": fire_name,
        "fire_year": FIRE_YEAR,
        "ignition_month": ignition_month,
        "fire_size_acres": fire_size_acres,

        "mtbs_valid_fraction": valid.min(1),
        "mtbs_unburned_low_fraction": unburned_low.min(1),
        "mtbs_low_fraction": low.min(1),
        "mtbs_moderate_fraction": moderate.min(1),
        "mtbs_high_fraction": high.min(1),
        "mtbs_increased_greenness_fraction": increased.min(1),
        "mtbs_nonmapping_fraction": nonmapping.min(1),

        "mtbs_meaningful_burn_fraction":
            meaningful_burn.min(1),
    })


patches_mtbs = patches_mtbs_raw.map(add_mtbs_summary)

In [25]:
# Sanity check
print(
    "Patches with MTBS attributes:",
    patches_mtbs.size().getInfo()
)

example_mtbs = ee.Feature(patches_mtbs.first())

fields = [
    "patch_id",
    "fire_overlap_fraction",
    "fire_id",
    "fire_name",
    "fire_year",
    "ignition_month",
    "fire_size_acres",
    "mtbs_valid_fraction",
    "mtbs_unburned_low_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "mtbs_increased_greenness_fraction",
    "mtbs_nonmapping_fraction",
    "mtbs_meaningful_burn_fraction",
]

for field in fields:
    print(
        field,
        ":",
        example_mtbs.get(field).getInfo()
    )

Patches with MTBS attributes: 45505
patch_id : HIGHLINE_7212_55640
fire_overlap_fraction : 1
fire_id : ID4549011522720170729
fire_name : HIGHLINE
fire_year : 2017
ignition_month : 7
fire_size_acres : 93699
mtbs_valid_fraction : 0.9625272331154684
mtbs_unburned_low_fraction : 0.24270152505446624
mtbs_low_fraction : 0.7198257080610021
mtbs_moderate_fraction : 0
mtbs_high_fraction : 0
mtbs_increased_greenness_fraction : 0
mtbs_nonmapping_fraction : 0
mtbs_meaningful_burn_fraction : 0.7198257080610021


#### Step 2A observations

MTBS focal-fire metadata and burn-severity information were successfully attached to all 45,505 retained Highline patches.

The example patch `HIGHLINE_7212_55640` is fully within the Highline perimeter and has approximately:

- 96.3% valid mapped MTBS severity coverage
- 24.3% unburned-to-low severity
- 72.0% low severity
- 0% moderate severity
- 0% high severity
- 0% increased greenness
- 0% non-mapping area
- 72.0% meaningful-burn fraction

`mtbs_valid_fraction` represents the portion of the patch assigned to mapped MTBS severity classes 1–5. MTBS class 6 is retained separately as `mtbs_nonmapping_fraction` so non-mapped areas are not accidentally treated as valid burn-severity observations.

The printed patch is only a sanity-check example; these MTBS attributes were calculated for all 45,505 retained 90 m patches.

### Step 2B — Summarize LANDFIRE pre-fire vegetation

The pre-fire LANDFIRE v1.4.0 vegetation snapshot is summarized onto the same fixed 90 m Highline patches already carrying MTBS attributes.

EVT defines whether each location belongs to a pre-fire forest/woodland ecosystem. Within that forested area, EVT composition, EVC tree-canopy cover, and EVH tree height are summarized geographically onto each patch.

The resulting patch-level vegetation features are:

- `forest_fraction`
- `dominant_evt`
- `dominant_evt_fraction`
- `tree_cover_mean`
- `tree_height_mean`

Valid EVC and EVH coverage within the EVT-defined forest area is also retained as QA information rather than treating missing structural measurements as zero.

In [30]:
# LANDFIRE v1.4.0 represents the pre-fire vegetation snapshot
# used for the 2017 Highline POC.

evt = ee.Image(
    ee.ImageCollection("LANDFIRE/Vegetation/EVT/v1_4_0")
    .filterBounds(highline_geom)
    .first()
).select("EVT")

evc = ee.Image(
    ee.ImageCollection("LANDFIRE/Vegetation/EVC/v1_4_0")
    .filterBounds(highline_geom)
    .first()
).select("EVC")

evh = ee.Image(
    ee.ImageCollection("LANDFIRE/Vegetation/EVH/v1_4_0")
    .filterBounds(highline_geom)
    .first()
).select("EVH")


# Highline forest/woodland EVT classes established in the LANDFIRE POC.
forest_evt_codes = [
    3056, 3233, 3050, 3235, 3055,
    3166, 3045, 3227, 3011, 3167,
    3053, 3046, 3234, 3159, 3047,
    3160, 3228, 3061, 3236, 3161,
]


# EVT defines forest/woodland eligibility.
forest_mask = (
    evt.remap(
        forest_evt_codes,
        [1] * len(forest_evt_codes),
        0,
    )
    .rename("forest_woodland")
)


# Convert EVC Tree Cover classes to midpoint percentages.
EVC_TREE_CODES = [
    101, 102, 103, 104, 105,
    106, 107, 108, 109,
]

EVC_TREE_COVER_VALUES = [
    15, 25, 35, 45, 55,
    65, 75, 85, 95,
]

tree_cover_pct = (
    evc.remap(
        EVC_TREE_CODES,
        EVC_TREE_COVER_VALUES,
    )
    .rename("tree_cover_pct")
    .updateMask(forest_mask.eq(1))
)


# LANDFIRE EVH tree-height coding:
# 101 = 1 m, 102 = 2 m, ..., 198 = 98 m.
tree_height_m = (
    evh
    .updateMask(
        evh.gte(101)
        .And(evh.lte(198))
        .And(forest_mask.eq(1))
    )
    .subtract(100)
    .rename("tree_height_m")
)


print("EVT scale:", evt.projection().nominalScale().getInfo())
print("EVC scale:", evc.projection().nominalScale().getInfo())
print("EVH scale:", evh.projection().nominalScale().getInfo())

print("Forest EVT classes:", len(forest_evt_codes))
print("EVC tree-cover codes:", EVC_TREE_CODES)
print("Tree-cover values (%):", EVC_TREE_COVER_VALUES)

print(
    "Tree-height coding:",
    "101–198 -> 1–98 m"
)

EVT scale: 30.000000000000004
EVC scale: 30
EVH scale: 30
Forest EVT classes: 20
EVC tree-cover codes: [101, 102, 103, 104, 105, 106, 107, 108, 109]
Tree-cover values (%): [15, 25, 35, 45, 55, 65, 75, 85, 95]
Tree-height coding: 101–198 -> 1–98 m


In [31]:
# Area classified as forest/woodland by EVT.
forest_area_image = (
    ee.Image.pixelArea()
    .updateMask(forest_mask.eq(1))
    .rename("forest_area_m2")
)

# Explicitly name the reducer output so the resulting feature property
# is guaranteed to be called forest_area_m2.
forest_area_reducer = (
    ee.Reducer.sum()
    .setOutputs(["forest_area_m2"])
)

patches_landfire = forest_area_image.reduceRegions(
    collection=patches_mtbs,
    reducer=forest_area_reducer,
    scale=30,
    crs=evt.projection(),
    tileScale=4,
)


def add_forest_fraction(feature):
    forest_area = safe_number(
        feature, "forest_area_m2"
    )

    return feature.set(
        "forest_fraction",
        forest_area.divide(PATCH_AREA_M2).min(1)
    )


patches_landfire = patches_landfire.map(
    add_forest_fraction
)

print(
    "Patches after EVT forest summary:",
    patches_landfire.size().getInfo()
)

example_landfire = ee.Feature(
    patches_landfire.first()
)

print(
    "Example patch:",
    example_landfire.get("patch_id").getInfo()
)

print(
    "Example forest fraction:",
    example_landfire.get("forest_fraction").getInfo()
)

print(
    "Mean forest fraction:",
    patches_landfire.aggregate_mean(
        "forest_fraction"
    ).getInfo()
)

print(
    "Minimum forest fraction:",
    patches_landfire.aggregate_min(
        "forest_fraction"
    ).getInfo()
)

print(
    "Maximum forest fraction:",
    patches_landfire.aggregate_max(
        "forest_fraction"
    ).getInfo()
)

print(
    "Patches with zero forest:",
    patches_landfire
    .filter(ee.Filter.eq("forest_fraction", 0))
    .size()
    .getInfo()
)

Patches after EVT forest summary: 45505
Example patch: HIGHLINE_7212_55640
Example forest fraction: 0.7816384269660797
Mean forest fraction: 0.9538525483687502
Minimum forest fraction: 0
Maximum forest fraction: 1
Patches with zero forest: 166


In [32]:
# Area of each patch with a valid numeric EVC tree-cover value.
tree_cover_valid_area_image = (
    ee.Image.pixelArea()
    .updateMask(tree_cover_pct.mask())
    .rename("tree_cover_valid_area_m2")
)

patches_landfire = tree_cover_valid_area_image.reduceRegions(
    collection=patches_landfire,
    reducer=ee.Reducer.sum().setOutputs(
        ["tree_cover_valid_area_m2"]
    ),
    scale=30,
    crs=evc.projection(),
    tileScale=4,
)


# Area-weighted tree-cover numerator:
# canopy-cover percentage × geographic area.
tree_cover_weighted_image = (
    ee.Image.pixelArea()
    .multiply(tree_cover_pct)
    .updateMask(tree_cover_pct.mask())
    .rename("tree_cover_weighted_sum")
)

patches_landfire = tree_cover_weighted_image.reduceRegions(
    collection=patches_landfire,
    reducer=ee.Reducer.sum().setOutputs(
        ["tree_cover_weighted_sum"]
    ),
    scale=30,
    crs=evc.projection(),
    tileScale=4,
)


def add_tree_cover_summary(feature):
    forest_area = safe_number(
        feature, "forest_area_m2"
    )

    valid_cover_area = safe_number(
        feature, "tree_cover_valid_area_m2"
    )

    weighted_cover = safe_number(
        feature, "tree_cover_weighted_sum"
    )

    # Fraction of EVT-defined forest that has usable EVC tree-cover data.
    valid_fraction = ee.Algorithms.If(
        forest_area.gt(0),
        valid_cover_area.divide(forest_area).min(1),
        None,
    )

    # Mean canopy cover over valid forested area only.
    tree_cover_mean = ee.Algorithms.If(
        valid_cover_area.gt(0),
        weighted_cover.divide(valid_cover_area),
        None,
    )

    return feature.set({
        "tree_cover_valid_fraction": valid_fraction,
        "tree_cover_mean": tree_cover_mean,
    })


patches_landfire = patches_landfire.map(
    add_tree_cover_summary
)


example_landfire = ee.Feature(
    patches_landfire.first()
)

print(
    "Patches after EVC summary:",
    patches_landfire.size().getInfo()
)

print(
    "Example patch:",
    example_landfire.get("patch_id").getInfo()
)

print(
    "Forest fraction:",
    example_landfire.get("forest_fraction").getInfo()
)

print(
    "Tree-cover valid fraction:",
    example_landfire.get(
        "tree_cover_valid_fraction"
    ).getInfo()
)

print(
    "Tree-cover mean (%):",
    example_landfire.get(
        "tree_cover_mean"
    ).getInfo()
)

Patches after EVC summary: 45505
Example patch: HIGHLINE_7212_55640
Forest fraction: 0.7816384269660797
Tree-cover valid fraction: 1
Tree-cover mean (%): 25.334750265674817


In [33]:
# Area of each patch with a valid numeric EVH tree-height value.
tree_height_valid_area_image = (
    ee.Image.pixelArea()
    .updateMask(tree_height_m.mask())
    .rename("tree_height_valid_area_m2")
)

patches_landfire = tree_height_valid_area_image.reduceRegions(
    collection=patches_landfire,
    reducer=ee.Reducer.sum().setOutputs(
        ["tree_height_valid_area_m2"]
    ),
    scale=30,
    crs=evh.projection(),
    tileScale=4,
)


# Area-weighted tree-height numerator:
# tree height (m) × geographic area.
tree_height_weighted_image = (
    ee.Image.pixelArea()
    .multiply(tree_height_m)
    .updateMask(tree_height_m.mask())
    .rename("tree_height_weighted_sum")
)

patches_landfire = tree_height_weighted_image.reduceRegions(
    collection=patches_landfire,
    reducer=ee.Reducer.sum().setOutputs(
        ["tree_height_weighted_sum"]
    ),
    scale=30,
    crs=evh.projection(),
    tileScale=4,
)


def add_tree_height_summary(feature):
    forest_area = safe_number(
        feature, "forest_area_m2"
    )

    valid_height_area = safe_number(
        feature, "tree_height_valid_area_m2"
    )

    weighted_height = safe_number(
        feature, "tree_height_weighted_sum"
    )

    tree_height_valid_fraction = ee.Algorithms.If(
        forest_area.gt(0),
        valid_height_area.divide(forest_area).min(1),
        None,
    )

    tree_height_mean = ee.Algorithms.If(
        valid_height_area.gt(0),
        weighted_height.divide(valid_height_area),
        None,
    )

    return feature.set({
        "tree_height_valid_fraction":
            tree_height_valid_fraction,

        "tree_height_mean":
            tree_height_mean,
    })


patches_landfire = patches_landfire.map(
    add_tree_height_summary
)


example_landfire = ee.Feature(
    patches_landfire.first()
)

print(
    "Patches after EVH summary:",
    patches_landfire.size().getInfo()
)

print(
    "Example patch:",
    example_landfire.get("patch_id").getInfo()
)

print(
    "Forest fraction:",
    example_landfire.get("forest_fraction").getInfo()
)

print(
    "Tree-height valid fraction:",
    example_landfire.get(
        "tree_height_valid_fraction"
    ).getInfo()
)

print(
    "Tree-height mean (m):",
    example_landfire.get(
        "tree_height_mean"
    ).getInfo()
)

Patches after EVH summary: 45505
Example patch: HIGHLINE_7212_55640
Forest fraction: 0.7816384269660797
Tree-height valid fraction: 1
Tree-height mean (m): 9.685441020191286


In [34]:
# For every forested EVT pixel, pair geographic area with its EVT code.
evt_area_image = ee.Image.cat([
    ee.Image.pixelArea()
        .updateMask(forest_mask.eq(1))
        .rename("evt_area_m2"),

    evt
        .updateMask(forest_mask.eq(1))
        .rename("evt_code"),
])


# Sum geographic area separately for each EVT class within each patch.
patches_landfire = evt_area_image.reduceRegions(
    collection=patches_landfire,
    reducer=(
        ee.Reducer.sum()
        .group(
            groupField=1,
            groupName="evt"
        )
    ),
    scale=30,
    crs=evt.projection(),
    tileScale=4,
)


def add_dominant_evt(feature):
    groups = ee.List(
        ee.Algorithms.If(
            feature.get("groups"),
            feature.get("groups"),
            ee.List([])
        )
    )

    forest_area = safe_number(
        feature, "forest_area_m2"
    )

    has_forest = groups.size().gt(0)

    # Convert grouped EVT-area dictionaries to a temporary
    # FeatureCollection so we can sort by area.
    grouped_evt = ee.FeatureCollection(
        groups.map(
            lambda group: ee.Feature(
                None,
                ee.Dictionary(group)
            )
        )
    )

    dominant_group = ee.Feature(
        ee.Algorithms.If(
            has_forest,
            grouped_evt.sort("sum", False).first(),
            ee.Feature(None, {
                "evt": None,
                "sum": 0
            })
        )
    )

    dominant_evt = dominant_group.get("evt")

    dominant_area = ee.Number(
        dominant_group.get("sum")
    )

    dominant_evt_fraction = ee.Algorithms.If(
        forest_area.gt(0),
        dominant_area.divide(forest_area).min(1),
        None,
    )

    return feature.set({
        "dominant_evt": dominant_evt,
        "dominant_evt_fraction":
            dominant_evt_fraction,
    })


patches_landfire = patches_landfire.map(
    add_dominant_evt
)


example_landfire = ee.Feature(
    patches_landfire.first()
)

print(
    "Patches after EVT composition summary:",
    patches_landfire.size().getInfo()
)

print(
    "Example patch:",
    example_landfire.get("patch_id").getInfo()
)

print(
    "Forest fraction:",
    example_landfire.get(
        "forest_fraction"
    ).getInfo()
)

print(
    "Dominant EVT:",
    example_landfire.get(
        "dominant_evt"
    ).getInfo()
)

print(
    "Dominant EVT fraction:",
    example_landfire.get(
        "dominant_evt_fraction"
    ).getInfo()
)

Patches after EVT composition summary: 45505
Example patch: HIGHLINE_7212_55640
Forest fraction: 0.7816384269660797
Dominant EVT: 3235
Dominant EVT fraction: 1


In [35]:
print(
    "Total patches:",
    patches_landfire.size().getInfo()
)

print(
    "Mean forest fraction:",
    patches_landfire.aggregate_mean(
        "forest_fraction"
    ).getInfo()
)

print(
    "Mean tree-cover valid fraction:",
    patches_landfire
    .filter(ee.Filter.gt("forest_fraction", 0))
    .aggregate_mean(
        "tree_cover_valid_fraction"
    )
    .getInfo()
)

print(
    "Mean tree-height valid fraction:",
    patches_landfire
    .filter(ee.Filter.gt("forest_fraction", 0))
    .aggregate_mean(
        "tree_height_valid_fraction"
    )
    .getInfo()
)

print(
    "Min dominant EVT fraction:",
    patches_landfire
    .filter(ee.Filter.gt("forest_fraction", 0))
    .aggregate_min(
        "dominant_evt_fraction"
    )
    .getInfo()
)

print(
    "Max dominant EVT fraction:",
    patches_landfire
    .filter(ee.Filter.gt("forest_fraction", 0))
    .aggregate_max(
        "dominant_evt_fraction"
    )
    .getInfo()
)

example_landfire = ee.Feature(
    patches_landfire.first()
)

final_fields = [
    "patch_id",
    "forest_fraction",
    "dominant_evt",
    "dominant_evt_fraction",
    "tree_cover_valid_fraction",
    "tree_cover_mean",
    "tree_height_valid_fraction",
    "tree_height_mean",
]

print("\nExample integrated LANDFIRE row:")

for field in final_fields:
    print(
        field,
        ":",
        example_landfire.get(field).getInfo()
    )

Total patches: 45505
Mean forest fraction: 0.9538525483687502
Mean tree-cover valid fraction: 0.9804218721500885
Mean tree-height valid fraction: 0.9907914261596672
Min dominant EVT fraction: 0.18574050745782086
Max dominant EVT fraction: 1

Example integrated LANDFIRE row:
patch_id : HIGHLINE_7212_55640
forest_fraction : 0.7816384269660797
dominant_evt : 3235
dominant_evt_fraction : 1
tree_cover_valid_fraction : 1
tree_cover_mean : 25.334750265674817
tree_height_valid_fraction : 1
tree_height_mean : 9.685441020191286


#### Step 2B observations

LANDFIRE vegetation information was successfully summarized onto all 45,505 retained Highline patches.

Across the retained 90 m grid:

- Mean pre-fire forest/woodland fraction is approximately 95.4%, closely matching the whole-fire LANDFIRE POC result.
- EVC tree-cover and EVH tree-height information have high valid coverage within EVT-defined forest.
- `dominant_evt_fraction` ranges from approximately 0.19 to 1.00, indicating that some patches contain multiple forest vegetation types while others are compositionally uniform.

For the example patch `HIGHLINE_7212_55640`:

- `forest_fraction` ≈ 0.782
- `dominant_evt` = 3235
- `dominant_evt_fraction` = 1.00
- `tree_cover_valid_fraction` = 1.00
- `tree_cover_mean` ≈ 25.3%
- `tree_height_valid_fraction` = 1.00
- `tree_height_mean` ≈ 9.69 m

Forest/woodland eligibility is defined by EVT. EVC and EVH provide canopy-cover and tree-height measurements only over their valid portions of that EVT-defined forest; missing structural information is tracked through coverage QA rather than treated as zero.

No minimum forest-fraction or EVC/EVH valid-coverage threshold is applied during the Highline integration POC. Those population-level decisions remain deferred to Phase 3.

### Step 2C — Summarize 3DEP terrain

USGS 3DEP terrain information is summarized onto the same fixed 90 m Highline patches already carrying MTBS and LANDFIRE attributes.

Elevation is taken from the native ~10 m 3DEP DEM. Slope and aspect are derived from the original unclipped elevation surface so that terrain calculations near patch and fire boundaries can still use neighboring elevation cells.

Raw aspect is converted to northness and eastness before aggregation because compass direction is circular.

The four core terrain features are:

- `elevation_mean`
- `slope_mean`
- `northness_mean`
- `eastness_mean`

Terrain summaries are calculated over the eligible pre-fire forest/woodland portion of each patch using geographic area weighting.

In [36]:
import math


# Load the ~10 m USGS 3DEP DEM covering Highline.
dep_collection = ee.ImageCollection(
    "USGS/3DEP/10m_collection"
)

dep_highline = dep_collection.filterBounds(
    highline_geom
)

elevation = ee.Image(
    dep_highline.first()
).select("elevation")


# Derive terrain products from the original DEM.
terrain = ee.Terrain.products(elevation)

slope = terrain.select("slope")
aspect = terrain.select("aspect")


# Convert circular aspect degrees into ML-friendly
# north-south and east-west directional features.
aspect_rad = aspect.multiply(
    math.pi / 180
)

northness = (
    aspect_rad
    .cos()
    .rename("northness")
)

eastness = (
    aspect_rad
    .sin()
    .rename("eastness")
)


print(
    "3DEP images intersecting Highline:",
    dep_highline.size().getInfo()
)

print(
    "Elevation bands:",
    elevation.bandNames().getInfo()
)

print(
    "Elevation nominal scale (m):",
    elevation.projection().nominalScale().getInfo()
)

print(
    "Terrain bands:",
    terrain.bandNames().getInfo()
)

print(
    "Northness band:",
    northness.bandNames().getInfo()
)

print(
    "Eastness band:",
    eastness.bandNames().getInfo()
)

3DEP images intersecting Highline: 1
Elevation bands: ['elevation']
Elevation nominal scale (m): 10.307360269724871
Terrain bands: ['elevation', 'slope', 'aspect', 'hillshade']
Northness band: ['northness']
Eastness band: ['eastness']


In [37]:
# Use the EVT-derived forest mask established in Step 2B.
# 3DEP values contribute only where the patch was pre-fire forest/woodland.

terrain_forest_mask = forest_mask.eq(1)

pixel_area_3dep = ee.Image.pixelArea()


# Forest area represented on the 3DEP processing grid.
terrain_forest_area = (
    pixel_area_3dep
    .updateMask(terrain_forest_mask)
    .rename("terrain_forest_area_m2")
)


# Area-weighted numerators for each terrain variable.
elevation_weighted = (
    elevation
    .multiply(pixel_area_3dep)
    .updateMask(terrain_forest_mask)
    .rename("elevation_weighted_sum")
)

slope_weighted = (
    slope
    .multiply(pixel_area_3dep)
    .updateMask(terrain_forest_mask)
    .rename("slope_weighted_sum")
)

northness_weighted = (
    northness
    .multiply(pixel_area_3dep)
    .updateMask(terrain_forest_mask)
    .rename("northness_weighted_sum")
)

eastness_weighted = (
    eastness
    .multiply(pixel_area_3dep)
    .updateMask(terrain_forest_mask)
    .rename("eastness_weighted_sum")
)


terrain_summary_image = ee.Image.cat([
    terrain_forest_area,
    elevation_weighted,
    slope_weighted,
    northness_weighted,
    eastness_weighted,
])


patches_terrain_raw = terrain_summary_image.reduceRegions(
    collection=patches_landfire,
    reducer=ee.Reducer.sum(),
    scale=10,
    crs=elevation.projection(),
    tileScale=4,
)


print(
    "Patches after 3DEP reduction:",
    patches_terrain_raw.size().getInfo()
)

print(
    "New reduction properties:",
    ee.Feature(
        patches_terrain_raw.first()
    ).toDictionary([
        "terrain_forest_area_m2",
        "elevation_weighted_sum",
        "slope_weighted_sum",
        "northness_weighted_sum",
        "eastness_weighted_sum",
    ]).getInfo()
)

Patches after 3DEP reduction: 45505
New reduction properties: {'eastness_weighted_sum': 5833.2373919108695, 'elevation_weighted_sum': 13138065.911447503, 'northness_weighted_sum': -2767.9023054797353, 'slope_weighted_sum': 235399.79026264863, 'terrain_forest_area_m2': 6559.77571554745}


In [38]:
def add_terrain_summary(feature):
    terrain_forest_area = safe_number(
        feature, "terrain_forest_area_m2"
    )

    elevation_sum = safe_number(
        feature, "elevation_weighted_sum"
    )

    slope_sum = safe_number(
        feature, "slope_weighted_sum"
    )

    northness_sum = safe_number(
        feature, "northness_weighted_sum"
    )

    eastness_sum = safe_number(
        feature, "eastness_weighted_sum"
    )

    elevation_mean = ee.Algorithms.If(
        terrain_forest_area.gt(0),
        elevation_sum.divide(terrain_forest_area),
        None,
    )

    slope_mean = ee.Algorithms.If(
        terrain_forest_area.gt(0),
        slope_sum.divide(terrain_forest_area),
        None,
    )

    northness_mean = ee.Algorithms.If(
        terrain_forest_area.gt(0),
        northness_sum.divide(terrain_forest_area),
        None,
    )

    eastness_mean = ee.Algorithms.If(
        terrain_forest_area.gt(0),
        eastness_sum.divide(terrain_forest_area),
        None,
    )

    return feature.set({
        "elevation_mean": elevation_mean,
        "slope_mean": slope_mean,
        "northness_mean": northness_mean,
        "eastness_mean": eastness_mean,
    })


patches_terrain = patches_terrain_raw.map(
    add_terrain_summary
)


example_terrain = ee.Feature(
    patches_terrain.first()
)

print(
    "Patches with terrain features:",
    patches_terrain.size().getInfo()
)

print(
    "Example patch:",
    example_terrain.get("patch_id").getInfo()
)

print(
    "Elevation mean (m):",
    example_terrain.get("elevation_mean").getInfo()
)

print(
    "Slope mean (degrees):",
    example_terrain.get("slope_mean").getInfo()
)

print(
    "Northness mean:",
    example_terrain.get("northness_mean").getInfo()
)

print(
    "Eastness mean:",
    example_terrain.get("eastness_mean").getInfo()
)

Patches with terrain features: 45505
Example patch: HIGHLINE_7212_55640
Elevation mean (m): 2002.8224258199439
Slope mean (degrees): 35.885341278471316
Northness mean: -0.42195075342583394
Eastness mean: 0.8892434200281272


In [39]:
terrain_patches = patches_terrain.filter(
    ee.Filter.gt("terrain_forest_area_m2", 0)
)

print(
    "Total patches:",
    patches_terrain.size().getInfo()
)

print(
    "Patches with forest terrain:",
    terrain_patches.size().getInfo()
)

print(
    "Mean elevation (m):",
    terrain_patches.aggregate_mean(
        "elevation_mean"
    ).getInfo()
)

print(
    "Mean slope (degrees):",
    terrain_patches.aggregate_mean(
        "slope_mean"
    ).getInfo()
)

print(
    "Elevation range (m):",
    terrain_patches.aggregate_min(
        "elevation_mean"
    ).getInfo(),
    "to",
    terrain_patches.aggregate_max(
        "elevation_mean"
    ).getInfo()
)

print(
    "Slope range (degrees):",
    terrain_patches.aggregate_min(
        "slope_mean"
    ).getInfo(),
    "to",
    terrain_patches.aggregate_max(
        "slope_mean"
    ).getInfo()
)

print(
    "Northness range:",
    terrain_patches.aggregate_min(
        "northness_mean"
    ).getInfo(),
    "to",
    terrain_patches.aggregate_max(
        "northness_mean"
    ).getInfo()
)

print(
    "Eastness range:",
    terrain_patches.aggregate_min(
        "eastness_mean"
    ).getInfo(),
    "to",
    terrain_patches.aggregate_max(
        "eastness_mean"
    ).getInfo()
)

Total patches: 45505
Patches with forest terrain: 45342
Mean elevation (m): 1989.4358050617736
Mean slope (degrees): 13.509659050409494
Elevation range (m): 1484.2134543610725 to 2541.7177155463673
Slope range (degrees): 0 to 44.63780440121035
Northness range: -0.9996522202533261 to 1
Eastness range: -0.9996719751741225 to 0.9999201023771234


#### Step 2C observations

USGS 3DEP terrain information was successfully summarized onto all 45,505 retained Highline patches.

Among patches containing eligible pre-fire forest terrain:

- Mean elevation is approximately 1,989 m.
- Mean slope is approximately 13.5°.
- Patch-level mean elevation ranges from approximately 1,484 m to 2,542 m.
- Patch-level mean slope ranges from approximately 0° to 44.6°.
- Mean northness and eastness remain within their expected `[-1, 1]` ranges.

The 90 m terrain summaries are calculated from the native ~10 m 3DEP surface over the EVT-defined forest/woodland portion of each patch. Elevation, slope, northness, and eastness are summarized using geographic area-weighted means.

The small difference between LANDFIRE-defined forest-patch counts and 3DEP forest-terrain counts reflects native-grid alignment and boundary effects between the separate raster products. Final production aggregation details remain subject to Phase 3 validation.

## Step 2 — Final Integration QA

Before moving to the temporal data sources, perform a final integrity check on the common patch dataset.

The checks verify that:

- the original 45,505 modeling patches are preserved with unique IDs;
- fraction-based features remain within valid `[0, 1]` ranges;
- the integrated MTBS severity composition reproduces the broad Highline pattern observed in the original MTBS POC;
- the final dataset still contains the expected MTBS, LANDFIRE, and 3DEP feature groups.

This is an integration-level sanity check rather than a substitute for the broader multi-fire validation that will occur in Phase 3.

In [39]:
# 1. Row-count and patch-ID integrity

total_patches = patches_terrain.size().getInfo()

unique_patch_ids = (
    patches_terrain
    .aggregate_count_distinct("patch_id")
    .getInfo()
)

print("PATCH INTEGRITY")
print("Total patches:", total_patches)
print("Unique patch IDs:", unique_patch_ids)
print("Duplicate patch IDs:", total_patches - unique_patch_ids)


# 2. Fraction-field range checks

fraction_fields = [
    "fire_overlap_fraction",
    "mtbs_valid_fraction",
    "mtbs_unburned_low_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "mtbs_increased_greenness_fraction",
    "mtbs_nonmapping_fraction",
    "mtbs_meaningful_burn_fraction",
    "forest_fraction",
    "dominant_evt_fraction",
    "tree_cover_valid_fraction",
    "tree_height_valid_fraction",
]

print("\nFRACTION RANGES")

for field in fraction_fields:
    field_min = (
        patches_terrain
        .aggregate_min(field)
        .getInfo()
    )

    field_max = (
        patches_terrain
        .aggregate_max(field)
        .getInfo()
    )

    print(
        f"{field}: "
        f"{field_min} to {field_max}"
    )


# 3. MTBS whole-grid severity sanity check

severity_fields = {
    "Unburned-to-low":
        "mtbs_unburned_low_fraction",
    "Low":
        "mtbs_low_fraction",
    "Moderate":
        "mtbs_moderate_fraction",
    "High":
        "mtbs_high_fraction",
    "Increased greenness":
        "mtbs_increased_greenness_fraction",
}

severity_sums = {
    label: patches_terrain
    .aggregate_sum(field)
    .getInfo()
    for label, field in severity_fields.items()
}

mapped_total = sum(
    severity_sums.values()
)

print("\nMTBS SEVERITY COMPOSITION")
print(
    "Mean mapped MTBS coverage:",
    patches_terrain
    .aggregate_mean("mtbs_valid_fraction")
    .getInfo()
)

for label, value in severity_sums.items():
    pct = (
        100 * value / mapped_total
        if mapped_total > 0
        else None
    )

    print(
        f"{label}: {pct:.2f}%"
    )

print(
    "Mean non-mapping fraction:",
    patches_terrain
    .aggregate_mean(
        "mtbs_nonmapping_fraction"
    )
    .getInfo()
)

# 4. Confirm representative final-row fields

example_final = ee.Feature(
    patches_terrain.first()
)

final_fields = [
    "patch_id",

    # MTBS
    "fire_id",
    "fire_year",
    "mtbs_meaningful_burn_fraction",

    # LANDFIRE
    "forest_fraction",
    "dominant_evt",
    "dominant_evt_fraction",
    "tree_cover_mean",
    "tree_height_mean",

    # 3DEP
    "elevation_mean",
    "slope_mean",
    "northness_mean",
    "eastness_mean",
]

print("\nREPRESENTATIVE FINAL STEP-2 ROW")

for field in final_fields:
    print(
        field,
        ":",
        example_final.get(field).getInfo()
    )

PATCH INTEGRITY
Total patches: 45505
Unique patch IDs: 45505
Duplicate patch IDs: 0

FRACTION RANGES
fire_overlap_fraction: 0.9000099652044943 to 1
mtbs_valid_fraction: 0.6400871459694987 to 1
mtbs_unburned_low_fraction: 0 to 1
mtbs_low_fraction: 0 to 1
mtbs_moderate_fraction: 0 to 1
mtbs_high_fraction: 0 to 1
mtbs_increased_greenness_fraction: 0 to 1
mtbs_nonmapping_fraction: 0 to 0
mtbs_meaningful_burn_fraction: 0 to 1
forest_fraction: 0 to 1
dominant_evt_fraction: 0.18574050745782086 to 1
tree_cover_valid_fraction: 0 to 1
tree_height_valid_fraction: 0 to 1

MTBS SEVERITY COMPOSITION
Mean mapped MTBS coverage: 0.9974948573970258
Unburned-to-low: 13.36%
Low: 30.99%
Moderate: 32.38%
High: 23.05%
Increased greenness: 0.23%
Mean non-mapping fraction: 0

REPRESENTATIVE FINAL STEP-2 ROW
patch_id : HIGHLINE_7212_55640
fire_id : ID4549011522720170729
fire_year : 2017
mtbs_meaningful_burn_fraction : 0.7198257080610021
forest_fraction : 0.7816384269660797
dominant_evt : 3235
dominant_evt_fract

#### Step 2 QA observations

The integrated Step 2 dataset passed the final integrity checks.

- All 45,505 retained modeling patches are present.
- All 45,505 `patch_id` values are unique.
- Fraction-based features remain within their expected `[0, 1]` ranges.
- Mean mapped MTBS coverage is approximately 99.75%.
- The integrated patch-level MTBS severity composition closely reproduces the original whole-fire Highline severity distribution.
- Representative rows retain the expected MTBS, LANDFIRE, and 3DEP feature groups.

The close agreement between the original fire-level POCs and the 90 m integration results provides evidence that the source-specific aggregation logic is behaving as intended.

A small number of patches have lower MTBS valid coverage despite high fire-perimeter overlap. No threshold is imposed during the Highline POC; the full production dataset will be used in Phase 3 to determine whether a minimum MTBS valid-coverage requirement is necessary.

## Step 2 — Conceptual Recap

The common 90 m × 90 m Highline grid is the modeling unit. Each retained patch remains the same geographic square while source-specific information is summarized onto it.

- **MTBS** adds focal-fire metadata and patch-level burn-severity composition.
- **LANDFIRE** describes pre-fire vegetation:
  - EVT defines forest/woodland eligibility and vegetation type.
  - EVC provides tree-canopy cover.
  - EVH provides tree height.
- **3DEP** describes terrain through elevation, slope, northness, and eastness.

Source rasters do not need to share the 90 m grid or align perfectly with one another. Their values are summarized according to geographic overlap with each fixed patch, using area-weighted calculations where appropriate.

LANDFIRE and 3DEP structural/terrain summaries are tied to the EVT-defined pre-fire forest/woodland portion of each patch so that these predictors describe the landscape relevant to forest recovery.

The result is still one row per 90 m patch, with progressively more fire, vegetation, and terrain attributes attached.

## Step 2 — Phase 3 Handoff

The Highline integration establishes the common spatial framework for the production dataset.

### Locked integration logic

- Modeling unit: fixed 90 m × 90 m patch.
- Grid projection: UTM Zone 11N (`EPSG:32611`) for the Highline POC.
- POC patches are retained when at least 90% of the cell overlaps the focal-fire perimeter.
- Native source rasters remain on their own grids and are summarized geographically onto the common modeling patches.
- Categorical rasters are represented through class areas/composition rather than numeric averaging.
- Continuous variables are summarized using geographic area-weighted calculations where appropriate.
- EVT defines pre-fire forest/woodland eligibility.
- EVC/EVH missing structural information within EVT-defined forest is tracked as coverage QA rather than treated as zero.
- 3DEP terrain features are summarized over the EVT-defined forested portion of each patch.
- Raw aspect is represented through northness and eastness rather than directly as 0–360° degrees.
- MTBS non-mapping is tracked separately from valid burn-severity classes.

### Decisions intentionally deferred to Phase 3

- Final fire-perimeter overlap threshold.
- Minimum MTBS valid-coverage requirement.
- Minimum `forest_fraction` required for model eligibility.
- Minimum valid EVC/EVH coverage requirements.
- Generalization of the EVT forest/woodland taxonomy across Western U.S. LANDFIRE versions.
- Historical LANDFIRE version selection and harmonization.
- Handling of rare cross-source boundary disagreements.
- Whether additional vegetation or terrain heterogeneity features provide useful predictive value.

Phase 3 will generalize this validated Highline logic into a reusable, batched multi-fire dataset pipeline rather than redesigning the underlying feature definitions.

### Production engineering note

The final Step 2 Earth Engine QA required approximately 28 minutes because repeated `.getInfo()` calls triggered evaluation of a long server-side computation graph. Phase 3 should avoid this pattern: expensive geospatial work should be batched and materialized/exported efficiently, while repeated table-level validation should primarily occur downstream using local/tabular tools such as Parquet and DuckDB.

## Step 3 — Attach Landsat and climate (Daymet) information

The fixed 90 m Highline patches now receive the time-varying information needed for the recovery prediction problem.

Step 3 integrates:

- **Landsat** spectral condition before the fire, after the first complete post-fire growing season, and during the fifth post-fire growing season.
- **Daymet** historical and early post-fire climate information.

The temporal variables are summarized onto the same modeling patches already carrying MTBS, LANDFIRE, and 3DEP attributes. Temporal cutoffs are preserved so that predictors available through the end of Season 1 remain distinct from the future Season 5 prediction target.

### Step 3A — Summarize Landsat NBR

The validated Landsat POC produced three core 30 m seasonal NBR rasters:

- `NBR_pre` — robust pre-fire baseline from the 2014–2016 growing seasons
- `NBR_1` — first complete post-fire growing season, 2018
- `NBR_5` — fifth post-fire growing season, 2022

`NBR_pre` and `NBR_1` are available by the primary prediction cutoff. `NBR_5` is retained as the future prediction target.

The POC materialized these rasters as Earth Engine assets on a common 30 m `EPSG:32611` grid after validating Landsat QA masking, surface-reflectance scaling, seasonal compositing, coverage, and projection behavior. The integration POC therefore reuses those validated stored rasters rather than reconstructing the full Landsat processing graph.

Patch-level Landsat summaries will be attached to the existing 90 m modeling units, with coverage retained as QA information.

In [40]:
# Load the validated/materialized Landsat POC outputs

LANDSAT_ASSET_ROOT = "projects/ee-wildfire-recovery-vasant/assets"

nbr_pre_saved = (
    ee.Image(f"{LANDSAT_ASSET_ROOT}/highline_nbr_pre")
    .select("NBR_pre")
)

nbr_1_saved = (
    ee.Image(f"{LANDSAT_ASSET_ROOT}/highline_nbr_1")
    .select("NBR_1")
)

nbr_5_saved = (
    ee.Image(f"{LANDSAT_ASSET_ROOT}/highline_nbr_5")
    .select("NBR_5")
)

for name, image in {
    "NBR_pre": nbr_pre_saved,
    "NBR_1": nbr_1_saved,
    "NBR_5": nbr_5_saved,
}.items():

    projection = image.projection()

    print(name)
    print("  Bands:", image.bandNames().getInfo())
    print("  CRS:", projection.crs().getInfo())
    print("  Nominal scale:", projection.nominalScale().getInfo())

NBR_pre
  Bands: ['NBR_pre']
  CRS: EPSG:32611
  Nominal scale: 30
NBR_1
  Bands: ['NBR_1']
  CRS: EPSG:32611
  Nominal scale: 30
NBR_5
  Bands: ['NBR_5']
  CRS: EPSG:32611
  Nominal scale: 30


In [41]:
# Aggregate Landsat onto patches
# Summarize the three 30 m Landsat NBR rasters
# onto the fixed 90 m modeling patches

pixel_area_landsat = ee.Image.pixelArea()

landsat_summary_image = ee.Image.cat([

    # NBR_pre
    pixel_area_landsat
    .updateMask(nbr_pre_saved.mask())
    .rename("nbr_pre_valid_area_m2"),

    nbr_pre_saved
    .multiply(pixel_area_landsat)
    .rename("nbr_pre_weighted_sum"),

    # NBR_1
    pixel_area_landsat
    .updateMask(nbr_1_saved.mask())
    .rename("nbr_1_valid_area_m2"),

    nbr_1_saved
    .multiply(pixel_area_landsat)
    .rename("nbr_1_weighted_sum"),

    # NBR_5
    pixel_area_landsat
    .updateMask(nbr_5_saved.mask())
    .rename("nbr_5_valid_area_m2"),

    nbr_5_saved
    .multiply(pixel_area_landsat)
    .rename("nbr_5_weighted_sum"),
])

patches_landsat_raw = landsat_summary_image.reduceRegions(
    collection=patches_terrain,
    reducer=ee.Reducer.sum(),
    scale=30,
    crs=nbr_pre_saved.projection(),
    tileScale=4,
)

print("Patches after Landsat reduction:", patches_landsat_raw.size().getInfo())

example_landsat_raw = ee.Feature(patches_landsat_raw.first())

for field in [
    "patch_id",
    "nbr_pre_valid_area_m2",
    "nbr_pre_weighted_sum",
    "nbr_1_valid_area_m2",
    "nbr_1_weighted_sum",
    "nbr_5_valid_area_m2",
    "nbr_5_weighted_sum",
]:
    print(field, ":", example_landsat_raw.get(field).getInfo())

Patches after Landsat reduction: 45505
patch_id : HIGHLINE_7212_55640
nbr_pre_valid_area_m2 : 8102.050598144531
nbr_pre_weighted_sum : 1498.4099712694047
nbr_1_valid_area_m2 : 8102.050598144531
nbr_1_weighted_sum : 720.6909489790656
nbr_5_valid_area_m2 : 8102.050598144531
nbr_5_weighted_sum : 1681.2880606494546


#### Step 3A.2 observations

The three validated 30 m Landsat NBR rasters were successfully reduced onto all 45,505 fixed 90 m Highline patches.

For each Landsat period, the integration retains:

- geographic area with valid NBR coverage;
- the area-weighted NBR sum needed to calculate the patch-level mean.

The example patch has essentially complete valid coverage for `NBR_pre`, `NBR_1`, and `NBR_5`.

The reported valid area is slightly above the nominal 8,100 m² patch area because `ee.Image.pixelArea()` measures geographic surface area rather than assuming an exact planar 90 m × 90 m area. Patch-level coverage will therefore be calculated relative to each patch's geographic area.

Masked Landsat observations remain missing and are not treated as zero.

In [42]:
# Create the actual patch-level Landsat features
# Convert Landsat area totals into patch-level NBR means and valid-coverage fractions

def add_landsat_features(feature):

    geographic_patch_area = feature.geometry().area(maxError=1)

    pre_valid = safe_number(feature, "nbr_pre_valid_area_m2")
    pre_weighted = safe_number(feature, "nbr_pre_weighted_sum")

    nbr1_valid = safe_number(feature, "nbr_1_valid_area_m2")
    nbr1_weighted = safe_number(feature, "nbr_1_weighted_sum")

    nbr5_valid = safe_number(feature, "nbr_5_valid_area_m2")
    nbr5_weighted = safe_number(feature, "nbr_5_weighted_sum")

    nbr_pre_mean = ee.Algorithms.If(
        pre_valid.gt(0),
        pre_weighted.divide(pre_valid),
        None,
    )

    nbr_1_mean = ee.Algorithms.If(
        nbr1_valid.gt(0),
        nbr1_weighted.divide(nbr1_valid),
        None,
    )

    nbr_5_mean = ee.Algorithms.If(
        nbr5_valid.gt(0),
        nbr5_weighted.divide(nbr5_valid),
        None,
    )

    return feature.set({
        "nbr_pre_mean": nbr_pre_mean,
        "nbr_pre_valid_fraction": pre_valid.divide(geographic_patch_area).min(1),

        "nbr_1_mean": nbr_1_mean,
        "nbr_1_valid_fraction": nbr1_valid.divide(geographic_patch_area).min(1),

        "nbr_5_mean": nbr_5_mean,
        "nbr_5_valid_fraction": nbr5_valid.divide(geographic_patch_area).min(1),
    })


patches_landsat = patches_landsat_raw.map(add_landsat_features)

example_landsat = ee.Feature(patches_landsat.first())

print("Patch:", example_landsat.get("patch_id").getInfo())

for field in [
    "nbr_pre_mean",
    "nbr_pre_valid_fraction",
    "nbr_1_mean",
    "nbr_1_valid_fraction",
    "nbr_5_mean",
    "nbr_5_valid_fraction",
]:
    print(field, ":", example_landsat.get(field).getInfo())

Patch: HIGHLINE_7212_55640
nbr_pre_mean : 0.18494206535967067
nbr_pre_valid_fraction : 1
nbr_1_mean : 0.08895167220309791
nbr_1_valid_fraction : 1
nbr_5_mean : 0.20751389296858874
nbr_5_valid_fraction : 1


#### Step 3A.3 observations

Area-weighted patch-level Landsat NBR features were successfully created from the 30 m seasonal rasters.

For the example patch `HIGHLINE_7212_55640`:

- `nbr_pre_mean` ≈ 0.185
- `nbr_1_mean` ≈ 0.089
- `nbr_5_mean` ≈ 0.208
- valid coverage is 100% for all three periods.

This patch therefore shows an initial post-fire NBR decline followed by recovery to slightly above its pre-fire NBR by Season 5.

Each NBR mean is calculated only over valid Landsat area within the patch, while the corresponding valid fraction separately tracks coverage.

In [43]:
# Add the Season 1 burn-drop feature (the main derived Landsat predictor)
# Add the pre-fire → Season 1 NBR decline

def add_landsat_derived_features(feature):

    nbr_pre = feature.get("nbr_pre_mean")
    nbr_1 = feature.get("nbr_1_mean")

    nbr_drop = ee.Algorithms.If(
        ee.Algorithms.IsEqual(nbr_pre, None),
        None,
        ee.Algorithms.If(
            ee.Algorithms.IsEqual(nbr_1, None),
            None,
            ee.Number(nbr_pre).subtract(ee.Number(nbr_1)),
        ),
    )

    return feature.set({
        "nbr_drop": nbr_drop,
    })


patches_landsat = patches_landsat.map(
    add_landsat_derived_features
)

example_landsat = ee.Feature(
    patches_landsat.first()
)

print(
    "Patch:",
    example_landsat.get("patch_id").getInfo()
)

for field in [
    "nbr_pre_mean",
    "nbr_1_mean",
    "nbr_drop",
    "nbr_5_mean",
]:
    print(
        field,
        ":",
        example_landsat.get(field).getInfo()
    )

Patch: HIGHLINE_7212_55640
nbr_pre_mean : 0.18494206535967067
nbr_1_mean : 0.08895167220309791
nbr_drop : 0.09599039315657276
nbr_5_mean : 0.20751389296858874


#### Step 3A.4 observations

The end-of-Season-1 Landsat change feature was successfully created:

`nbr_drop = nbr_pre_mean - nbr_1_mean`

For the example patch `HIGHLINE_7212_55640`:

- `nbr_pre_mean` ≈ 0.185
- `nbr_1_mean` ≈ 0.089
- `nbr_drop` ≈ 0.096
- `nbr_5_mean` ≈ 0.208

A positive `nbr_drop` indicates that NBR declined from the pre-fire baseline to the first complete post-fire growing season.

`nbr_drop` is available by the primary prediction cutoff and may be used as a predictor. `nbr_5_mean` represents future Season 5 information and is retained as the prediction target rather than as a predictor.

In [44]:
# Compact Landsat integration sanity check

print("LANDSAT PATCH SANITY")

print("Total patches:", patches_landsat.size().getInfo())
print(
    "Unique patch IDs:",
    patches_landsat.aggregate_count_distinct("patch_id").getInfo()
)

landsat_fields = [
    "nbr_pre_mean",
    "nbr_1_mean",
    "nbr_5_mean",
    "nbr_drop",
]

for field in landsat_fields:
    stats = patches_landsat.aggregate_stats(field).getInfo()

    print(f"\n{field}")
    print("  mean:", stats["mean"])
    print("  min:", stats["min"])
    print("  max:", stats["max"])


print("\nMEAN VALID COVERAGE")

for field in [
    "nbr_pre_valid_fraction",
    "nbr_1_valid_fraction",
    "nbr_5_valid_fraction",
]:
    print(
        field,
        ":",
        patches_landsat.aggregate_mean(field).getInfo()
    )

LANDSAT PATCH SANITY
Total patches: 45505
Unique patch IDs: 45505

nbr_pre_mean
  mean: 0.3007106968611744
  min: -0.233945357555557
  max: 0.7207667745399656

nbr_1_mean
  mean: 0.0021024543837979102
  min: -0.5160175605184998
  max: 0.6870164785961183

nbr_5_mean
  mean: 0.12140966001231605
  min: -0.29223982122875064
  max: 0.7208834287951073

nbr_drop
  mean: 0.2986082424773767
  min: -0.24575657636127657
  max: 1.0804921919665573

MEAN VALID COVERAGE
nbr_pre_valid_fraction : 0.9994091448254622
nbr_1_valid_fraction : 0.9994091448254603
nbr_5_valid_fraction : 0.9994091448254622


#### Step 3A.5 observations

The Landsat integration passed the patch-level sanity checks.

- All 45,505 modeling patches remain present with unique `patch_id` values.
- Mean patch-level NBR is approximately:
  - `nbr_pre_mean` = 0.301
  - `nbr_1_mean` = 0.002
  - `nbr_5_mean` = 0.121
- This reproduces the expected broad pattern of strong post-fire decline followed by partial recovery by Season 5.
- Mean valid Landsat coverage is approximately 99.94% for all three periods.
- `nbr_drop` ranges from approximately -0.25 to 1.08. Because it is the difference between two NBR values, its valid range is wider than the individual NBR range.

No Landsat coverage threshold is imposed during the Highline integration POC; final coverage requirements remain a production-level data-quality decision.

### Step 3A — Landsat integration recap

Validated 30 m Landsat seasonal NBR products were successfully integrated onto the common 90 m Highline modeling grid.

Each patch now contains:

- `nbr_pre_mean` — pre-fire NBR baseline
- `nbr_1_mean` — first post-fire growing-season NBR
- `nbr_drop` — pre-fire to Season 1 NBR decline
- `nbr_5_mean` — fifth post-fire growing-season NBR prediction target
- valid-coverage fractions for each Landsat period

`nbr_pre_mean`, `nbr_1_mean`, and `nbr_drop` are available by the primary prediction cutoff. `nbr_5_mean` is future information and is retained only as the target.

The original Landsat POC's validated materialized rasters were reused, avoiding reconstruction of the deeper Landsat preprocessing graph during integration.

### Step 3B — Summarize Daymet climate

Daymet V4 provides daily climate information on a native ~1 km grid. Climate features are first aggregated temporally at the Daymet-grid level and are then geographically summarized onto the fixed 90 m modeling patches.

The integration uses the 16 core Daymet features established by the source POC:

- historical June–September precipitation, temperature, and VPD;
- actual January–May antecedent precipitation, temperature, and VPD;
- antecedent anomalies relative to matching historical conditions;
- Season-1 anomalies relative to the historical June–September baseline.

For Highline, the historical reference uses the 10 complete pre-fire years from 2007–2016. The antecedent period is January–May 2018 and Season 1 is June–September 2018.

Daymet remains a coarse-scale climate source: association with 90 m patches preserves geographic variation but does not create new 90 m climate information.

In [9]:
# Load Daymet and add daily vapor pressure deficit (VPD)

daymet = (
    ee.ImageCollection("NASA/ORNL/DAYMET_V4")
    .filterBounds(highline_geom)
)


def add_daymet_vpd(image):

    tmax = image.select("tmax")
    tmin = image.select("tmin")

    # Saturation vapor pressure at Tmax and Tmin (kPa)
    es_tmax = (
        tmax.multiply(17.27)
        .divide(tmax.add(237.3))
        .exp()
        .multiply(0.6108)
    )

    es_tmin = (
        tmin.multiply(17.27)
        .divide(tmin.add(237.3))
        .exp()
        .multiply(0.6108)
    )

    es_mean = es_tmax.add(es_tmin).divide(2)

    # Daymet vapor pressure is stored in Pa → convert to kPa
    actual_vp = image.select("vp").divide(1000)

    vpd = (
        es_mean
        .subtract(actual_vp)
        .max(0)
        .rename("vpd_kpa")
    )

    return image.addBands(vpd)


daymet_vpd = daymet.map(add_daymet_vpd)

sample_daymet = ee.Image(daymet_vpd.first())

print(
    "Daymet bands:",
    sample_daymet.bandNames().getInfo()
)

print(
    "Native nominal scale:",
    sample_daymet.select("tmax")
    .projection()
    .nominalScale()
    .getInfo()
)

print(
    "Available dates:",
    ee.Date(daymet.aggregate_min("system:time_start"))
    .format("YYYY-MM-dd")
    .getInfo(),
    "to",
    ee.Date(daymet.aggregate_max("system:time_start"))
    .format("YYYY-MM-dd")
    .getInfo(),
)

Daymet bands: ['dayl', 'prcp', 'srad', 'swe', 'tmax', 'tmin', 'vp', 'vpd_kpa']
Native nominal scale: 1000
Available dates: 1980-01-01 to 2025-12-31


#### Step 3B.1 observations

Daymet V4 loaded successfully and the daily vapor-pressure-deficit (`vpd_kpa`) band was added.

The working collection contains the seven native Daymet variables plus derived VPD and retains Daymet's native 1,000 m grid.

The available record spans 1980–2025, providing sufficient coverage for Highline's 10-year pre-fire historical baselines and the 2018 antecedent and Season-1 periods.

In [10]:
# Build the 16 core Daymet climate surfaces

def daymet_period_image(start_date, end_date):
    """
    Create one climate-summary image for a specified period.
    Temporal aggregation occurs independently within each
    native Daymet grid cell.
    """

    period = daymet_vpd.filterDate(
        start_date,
        end_date,
    )

    return ee.Image.cat([
        period.select("prcp")
        .sum()
        .rename("prcp_mm"),

        period.select("tmax")
        .mean()
        .rename("tmax_c"),

        period.select("tmin")
        .mean()
        .rename("tmin_c"),

        period.select("vpd_kpa")
        .mean()
        .rename("vpd_kpa"),
    ])


# 1. Historical Jun-Sep baseline: 2007-2016

historical_s1_images = ee.ImageCollection.fromImages([
    daymet_period_image(
        f"{year}-06-01",
        f"{year}-10-01",
    )
    for year in range(2007, 2017)
])

historical_s1 = historical_s1_images.mean()


# 2. Historical Jan-May baseline: 2007-2016 Used only for antecedent anomalies

historical_ante_images = ee.ImageCollection.fromImages([
    daymet_period_image(
        f"{year}-01-01",
        f"{year}-06-01",
    )
    for year in range(2007, 2017)
])

historical_ante = historical_ante_images.mean()



# 3. Actual 2018 antecedent and Season-1 climate

antecedent_2018 = daymet_period_image(
    "2018-01-01",
    "2018-06-01",
)

season1_2018 = daymet_period_image(
    "2018-06-01",
    "2018-10-01",
)


# 4. Assemble the 16 core primary-model surfaces

daymet_core_image = ee.Image.cat([

    # Historical Jun-Sep climate
    historical_s1.select("prcp_mm")
    .rename("hist_prcp_mm"),

    historical_s1.select("tmax_c")
    .rename("hist_tmax_c"),

    historical_s1.select("tmin_c")
    .rename("hist_tmin_c"),

    historical_s1.select("vpd_kpa")
    .rename("hist_s1_vpd_kpa"),


    # Actual antecedent Jan-May climate
    antecedent_2018.select("prcp_mm")
    .rename("ante_prcp_mm"),

    antecedent_2018.select("tmax_c")
    .rename("ante_tmax_c"),

    antecedent_2018.select("tmin_c")
    .rename("ante_tmin_c"),

    antecedent_2018.select("vpd_kpa")
    .rename("ante_vpd_kpa"),


    # Antecedent anomalies
    antecedent_2018.select("prcp_mm")
    .subtract(historical_ante.select("prcp_mm"))
    .rename("ante_prcp_anom_mm"),

    antecedent_2018.select("tmax_c")
    .subtract(historical_ante.select("tmax_c"))
    .rename("ante_tmax_anom_c"),

    antecedent_2018.select("tmin_c")
    .subtract(historical_ante.select("tmin_c"))
    .rename("ante_tmin_anom_c"),

    antecedent_2018.select("vpd_kpa")
    .subtract(historical_ante.select("vpd_kpa"))
    .rename("ante_vpd_anom_kpa"),


    # Season-1 anomalies
    season1_2018.select("prcp_mm")
    .subtract(historical_s1.select("prcp_mm"))
    .rename("s1_prcp_anom_mm"),

    season1_2018.select("tmax_c")
    .subtract(historical_s1.select("tmax_c"))
    .rename("s1_tmax_anom_c"),

    season1_2018.select("tmin_c")
    .subtract(historical_s1.select("tmin_c"))
    .rename("s1_tmin_anom_c"),

    season1_2018.select("vpd_kpa")
    .subtract(historical_s1.select("vpd_kpa"))
    .rename("s1_vpd_anom_kpa"),
])


print(
    "Core Daymet bands:",
    daymet_core_image.bandNames().getInfo()
)

print(
    "Number of core bands:",
    daymet_core_image.bandNames().size().getInfo()
)

print(
    "Nominal scale:",
    daymet_core_image
    .select("hist_prcp_mm")
    .projection()
    .nominalScale()
    .getInfo()
)

Core Daymet bands: ['hist_prcp_mm', 'hist_tmax_c', 'hist_tmin_c', 'hist_s1_vpd_kpa', 'ante_prcp_mm', 'ante_tmax_c', 'ante_tmin_c', 'ante_vpd_kpa', 'ante_prcp_anom_mm', 'ante_tmax_anom_c', 'ante_tmin_anom_c', 'ante_vpd_anom_kpa', 's1_prcp_anom_mm', 's1_tmax_anom_c', 's1_tmin_anom_c', 's1_vpd_anom_kpa']
Number of core bands: 16
Nominal scale: 111319.49079327357


In [11]:
# Restore Daymet's native 1 km projection to the temporally aggregated climate feature image

daymet_native_projection = (
    ee.Image(daymet.first())
    .select("tmax")
    .projection()
)

daymet_core_image = (
    daymet_core_image
    .setDefaultProjection(daymet_native_projection)
)

print("Native Daymet projection:")
print(daymet_native_projection.getInfo())

print(
    "\nCore-image nominal scale:",
    daymet_core_image
    .select("hist_prcp_mm")
    .projection()
    .nominalScale()
    .getInfo()
)

print(
    "Core-image band count:",
    daymet_core_image.bandNames().size().getInfo()
)

Native Daymet projection:
{'type': 'Projection', 'wkt': 'PROJCS["unnamed", \n  GEOGCS["NAD83", \n    DATUM["WGS_1984", \n      SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]], \n      AUTHORITY["EPSG","6326"]], \n    PRIMEM["Greenwich", 0.0], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH]], \n  PROJECTION["Lambert_Conformal_Conic_2SP"], \n  PARAMETER["central_meridian", -100.0], \n  PARAMETER["latitude_of_origin", 42.5], \n  PARAMETER["standard_parallel_1", 60.0], \n  PARAMETER["false_easting", 0.0], \n  PARAMETER["false_northing", 0.0], \n  PARAMETER["scale_factor", 1.0], \n  PARAMETER["standard_parallel_2", 25.0], \n  UNIT["m", 1.0], \n  AXIS["x", EAST], \n  AXIS["y", NORTH]]', 'transform': [1000, 0, -5802750, 0, -1000, 4984500]}

Core-image nominal scale: 1000
Core-image band count: 16


#### Step 3B.2 observations

The 16 core Daymet climate surfaces were constructed successfully from the daily record.

Temporal aggregation initially caused the resulting multi-year climate composite to fall back to Earth Engine's generic default projection. The native Daymet projection was therefore explicitly restored using the source image projection.

After correction:

- core Daymet feature bands = 16;
- native projection = Lambert Conformal Conic;
- nominal scale = 1,000 m.

The climate surfaces therefore retain Daymet's intended coarse spatial resolution before association with the 90 m modeling patches.

In [12]:
# Geographically summarize coarse Daymet climate onto the fixed 90 m patches

DAYMET_OVERLAP_SCALE = 30

daymet_core_names = [
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_tmin_c",
    "hist_s1_vpd_kpa",

    "ante_prcp_mm",
    "ante_tmax_c",
    "ante_tmin_c",
    "ante_vpd_kpa",

    "ante_prcp_anom_mm",
    "ante_tmax_anom_c",
    "ante_tmin_anom_c",
    "ante_vpd_anom_kpa",

    "s1_prcp_anom_mm",
    "s1_tmax_anom_c",
    "s1_tmin_anom_c",
    "s1_vpd_anom_kpa",
]


# Geographic area used for weighting
daymet_pixel_area = ee.Image.pixelArea()

daymet_valid_area = (
    daymet_pixel_area
    .updateMask(
        daymet_core_image
        .select("hist_prcp_mm")
        .mask()
    )
    .rename("daymet_valid_area_m2")
)


# value × geographic area for each climate feature
daymet_weighted = (
    daymet_core_image
    .multiply(daymet_pixel_area)
    .rename([
        f"{name}_weighted_sum"
        for name in daymet_core_names
    ])
)


daymet_summary_image = ee.Image.cat([
    daymet_valid_area,
    daymet_weighted,
])


# Perform the spatial reduction against the original
# common patch geometries rather than the deeper feature graph
patches_daymet_raw = daymet_summary_image.reduceRegions(
    collection=highline_patches,
    reducer=ee.Reducer.sum(),
    scale=DAYMET_OVERLAP_SCALE,
    crs=GRID_CRS,
    tileScale=4,
)


def make_daymet_patch_features(feature):

    valid_area = safe_number(
        feature,
        "daymet_valid_area_m2",
    )

    geographic_patch_area = (
        feature.geometry().area(maxError=1)
    )

    output = {
        "daymet_valid_fraction":
            valid_area
            .divide(geographic_patch_area)
            .min(1)
    }

    for name in daymet_core_names:

        weighted_sum = safe_number(
            feature,
            f"{name}_weighted_sum",
        )

        output[name] = ee.Algorithms.If(
            valid_area.gt(0),
            weighted_sum.divide(valid_area),
            None,
        )

    return feature.set(output)


daymet_patch_features = patches_daymet_raw.map(
    make_daymet_patch_features
)


example_daymet = ee.Feature(
    daymet_patch_features.first()
)

print(
    "Daymet patch rows:",
    daymet_patch_features.size().getInfo()
)

print(
    "Patch:",
    example_daymet.get("patch_id").getInfo()
)

for field in [
    "daymet_valid_fraction",
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_s1_vpd_kpa",
    "ante_prcp_mm",
    "ante_prcp_anom_mm",
    "s1_prcp_anom_mm",
    "s1_vpd_anom_kpa",
]:
    print(
        field,
        ":",
        example_daymet.get(field).getInfo()
    )

Daymet patch rows: 45505
Patch: HIGHLINE_7212_55640
daymet_valid_fraction : 1
hist_prcp_mm : 162.25400025248527
hist_tmax_c : 21.429073333740234
hist_s1_vpd_kpa : 1.0181050946981727
ante_prcp_mm : 550.2000007629395
ante_prcp_anom_mm : 113.68100201487539
s1_prcp_anom_mm : -51.654000943899156
s1_vpd_anom_kpa : -0.03468001607010729


#### Step 3B.3 observations

Daymet climate features were successfully associated with all 45,505 fixed 90 m Highline patches.

For the example patch `HIGHLINE_7212_55640`:

- Daymet valid coverage = 100%
- historical June–September precipitation ≈ 162.3 mm
- historical mean Tmax ≈ 21.4°C
- historical growing-season VPD ≈ 1.02 kPa
- antecedent January–May precipitation ≈ 550.2 mm
- antecedent precipitation anomaly ≈ +113.7 mm
- Season-1 precipitation anomaly ≈ −51.7 mm
- Season-1 VPD anomaly ≈ −0.035 kPa

These patch-level values differ from the Highline-wide POC averages, confirming that the coarse ~1 km Daymet surfaces retain spatial variation across the fire.

Association with 90 m patches does not imply 90 m climate resolution. Nearby patches may share identical or similar Daymet information, while patches crossing native Daymet-cell boundaries receive geographically weighted summaries.

#### Step 3B.4 — Daymet integration strategy

Daymet patch features are kept as a separate `FeatureCollection` keyed by `patch_id` rather than immediately joining them onto the existing MTBS + LANDFIRE + 3DEP + Landsat Earth Engine computation graph.

An attempted cross-source join during integration caused very slow evaluation because Earth Engine had to traverse the full upstream lazy computation graph for all 45,505 patches.

The Daymet integration has already established that:

- the same fixed 45,505 modeling patches are used;
- `patch_id` is preserved;
- the 16 core climate features can be calculated spatially for individual patches;
- Daymet coverage is complete for the inspected patch;
- coarse Daymet climate variation is retained without implying true 90 m climate resolution.

The source-specific collections will therefore remain separate during the POC and will be assembled deliberately during the final integration step rather than repeatedly deepening the Earth Engine graph.

In [13]:
# Compact Daymet QA
#
# Validate the climate surfaces directly rather than
# evaluating a full cross-source Earth Engine join.

daymet_check_fields = [
    "hist_prcp_mm",
    "hist_tmax_c",
    "ante_prcp_mm",
    "s1_prcp_anom_mm",
    "s1_vpd_anom_kpa",
]

daymet_firewide_check = (
    daymet_core_image
    .select(daymet_check_fields)
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        crs=daymet_native_projection,
        maxPixels=1e7,
    )
    .getInfo()
)

print("DAYMET COMPACT QA")
print("Patch rows previously verified:", 45505)

for field in daymet_check_fields:
    print(
        field,
        ":",
        daymet_firewide_check[field]
    )

DAYMET COMPACT QA
Patch rows previously verified: 45505
hist_prcp_mm : 170.126716502988
hist_tmax_c : 21.590490139460034
ante_prcp_mm : 579.0613595753298
s1_prcp_anom_mm : -39.83037361320219
s1_vpd_anom_kpa : -0.06156016904187501


#### Step 3B.4 observations

The compact Daymet QA reproduced the original Highline POC climate summaries very closely.

Representative fire-wide values include:

- historical June–September precipitation ≈ 170.1 mm;
- historical mean Tmax ≈ 21.59°C;
- antecedent January–May precipitation ≈ 579.1 mm;
- Season-1 precipitation anomaly ≈ −39.8 mm;
- Season-1 VPD anomaly ≈ −0.062 kPa.

Together with the previously verified 45,505 patch-level Daymet rows, these results confirm that the Daymet temporal feature construction and spatial association are behaving as intended.

Daymet remains stored separately from the deeper cross-source Earth Engine feature graph during the POC and will be assembled with the other patch-level source outputs during final integration.

### Step 3B — Daymet integration recap

Daymet V4 climate information was successfully converted from daily ~1 km observations into leakage-safe climate features associated with the common 90 m Highline patches.

The integration established:

- a rolling 10-complete-year pre-fire climate baseline;
- January–May antecedent post-fire climate;
- June–September Season-1 climate;
- calendar-matched climate anomalies;
- derived vapor pressure deficit (VPD);
- 16 core primary-model Daymet features.

Temporal aggregation occurs at Daymet's native coarse climate grid before spatial association with the 90 m modeling patches. This preserves Daymet as broad-scale climate context rather than implying true 90 m weather resolution.

The early-model climate contract remains restricted to the four pre-fire historical features, while the primary model may use the full 16-feature core set through the end-of-Season-1 cutoff.

### Step 3C — Verify temporal and cross-source consistency

Before moving to disturbance QA, verify that the Landsat and Daymet integrations conform to the same modeling-unit and information-cutoff contracts.

Both sources are keyed to the fixed Highline `patch_id` grid. Landsat provides pre-fire condition, Season-1 condition, and the future Season-5 target, while Daymet provides climate information only through the end of Season 1.

This check focuses on integration contracts rather than physically joining the two Earth Engine feature graphs.

In [14]:
# Step 3C — Cross-source and temporal contract sanity
#
# This is intentionally a lightweight contract check.
# It does NOT join or re-evaluate the large source graphs.

# Landsat model contract established in Step 3A
landsat_primary_predictors = [
    "nbr_pre_mean",
    "nbr_1_mean",
    "nbr_drop",
]

landsat_target = "nbr_5_mean"


# Daymet model contract established in Step 3B
daymet_primary_predictors = daymet_core_names

daymet_early_predictors = [
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_tmin_c",
    "hist_s1_vpd_kpa",
]

# Local assertions

assert len(daymet_primary_predictors) == 16

assert set(daymet_early_predictors).issubset(
    set(daymet_primary_predictors)
)

assert landsat_target not in landsat_primary_predictors

assert "nbr_5" not in " ".join(
    landsat_primary_predictors
).lower()

assert all(
    name.startswith(
        (
            "hist_",
            "ante_",
            "s1_",
        )
    )
    for name in daymet_primary_predictors
)


print("STEP 3C CONTRACT CHECK")

print("\nCommon modeling unit:")
print("90 m Highline patch keyed by patch_id")

print("\nPreviously verified patch count:")
print("45,505")

print("\nLandsat predictors:")
for name in landsat_primary_predictors:
    print(" -", name)

print("\nLandsat future target:")
print(" -", landsat_target)

print("\nDaymet primary predictors:")
print(" -", len(daymet_primary_predictors), "features")
print(" - information through end of Season 1 only")

print("\nDaymet early-model predictors:")
for name in daymet_early_predictors:
    print(" -", name)

print("\nContract checks passed.")

STEP 3C CONTRACT CHECK

Common modeling unit:
90 m Highline patch keyed by patch_id

Previously verified patch count:
45,505

Landsat predictors:
 - nbr_pre_mean
 - nbr_1_mean
 - nbr_drop

Landsat future target:
 - nbr_5_mean

Daymet primary predictors:
 - 16 features
 - information through end of Season 1 only

Daymet early-model predictors:
 - hist_prcp_mm
 - hist_tmax_c
 - hist_tmin_c
 - hist_s1_vpd_kpa

Contract checks passed.


#### Step 3C observations

The Landsat and Daymet integration contracts are consistent.

- Both sources use the same fixed 90 m Highline modeling unit identified by `patch_id`.
- Landsat primary predictors are restricted to `NBR_pre`, `NBR_1`, and `NBR_drop`.
- `NBR_5` remains isolated as the future prediction target.
- The 16 primary Daymet features contain climate information only through the end of Season 1.
- The early-model Daymet feature set is restricted to the four historical pre-fire climate variables.

No post-Season-1 climate information or Season-5 Landsat information is included in the predictor contract.

## Step 3 recap — Landsat and Daymet

Step 3 established the dynamic spectral and climate components of the modeling table.

### Landsat

Saved 30 m Landsat NBR assets were summarized geographically onto the common 90 m patches.

The resulting patch-level variables are:

- `nbr_pre_mean` — robust pre-fire spectral condition;
- `nbr_1_mean` — first complete post-fire growing-season condition;
- `nbr_drop` — focal-fire spectral change (`NBR_pre - NBR_1`);
- `nbr_5_mean` — fifth post-fire growing-season NBR and future model target.

Valid-data fractions were also retained for QA.

### Daymet

Daily Daymet V4 observations were temporally aggregated on the native ~1 km climate grid before being associated with the 90 m modeling patches.

The integration produced 16 primary climate predictors covering:

- historical pre-fire climate;
- antecedent post-fire climate;
- antecedent climate anomalies;
- Season-1 climate anomalies.

Derived VPD is included alongside precipitation and temperature.

Daymet remains a coarse-scale climate source; attaching its values to 90 m rows does not imply 90 m climate resolution.

### Integration strategy

Landsat had already been attached to the existing MTBS + LANDFIRE + 3DEP Earth Engine feature graph.

Daymet is temporarily retained as a separate patch-level collection keyed by `patch_id`. This avoids repeatedly extending an increasingly expensive lazy Earth Engine computation graph. The source-specific outputs will be assembled deliberately during the final integration step.

The temporal contract is now clear: all predictors contain information available no later than the end of Season 1, while `NBR_5` remains the future outcome.

## Step 4 — Disturbance QA

The final modeling population should represent recovery from the focal wildfire rather than recovery from other major disturbances.

### Step 4A — MTBS prior-fire and reburn screening

MTBS fire history is used to identify two patch-level conditions:

1. **Recent prior fire** — another wildfire occurred sufficiently recently before the focal event that the pre-fire baseline may already represent disturbance recovery.
2. **Reburn** — another wildfire affected the patch after the focal event but before the end of Season 5.

For the Highline POC, a 7-year prior-fire lookback is used as a provisional screening window. The final production lookback will be evaluated later rather than treated as permanently fixed.

For the 2017 Highline focal fire:

- prior-fire window: 2010–2016;
- reburn window: 2018–2022.

These indicators are eligibility/QA variables rather than predictive features.

In [ ]:
# Step 4A.1 — Define MTBS disturbance-screening windows

FOCAL_FIRE_YEAR = 2017

PRIOR_LOOKBACK_YEARS = 7

PRIOR_START_YEAR = (
    FOCAL_FIRE_YEAR - PRIOR_LOOKBACK_YEARS
)

PRIOR_END_YEAR = (
    FOCAL_FIRE_YEAR - 1
)

SEASON5_YEAR = (
    FOCAL_FIRE_YEAR + 5
)

REBURN_START_YEAR = (
    FOCAL_FIRE_YEAR + 1
)

REBURN_END_YEAR = SEASON5_YEAR


print("MTBS DISTURBANCE WINDOWS")

print(
    "Prior-fire window:",
    PRIOR_START_YEAR,
    "through",
    PRIOR_END_YEAR,
)

print(
    "Reburn window:",
    REBURN_START_YEAR,
    "through",
    REBURN_END_YEAR,
)

MTBS DISTURBANCE WINDOWS
Prior-fire window: 2010 through 2016
Reburn window: 2018 through 2022


#### Step 4A.1 observations

The Highline disturbance-screening windows are:

- recent prior-fire window: 2010–2016;
- reburn window: 2018–2022.

The 7-year prior-fire lookback is provisional for the POC. The production lookback remains an empirical Phase 3 decision.

In [ ]:
# Step 4A.2 — Build MTBS prior-fire and reburn masks

mtbs_boundaries = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

# Explicit millisecond date boundaries
prior_start_ms = (
    ee.Date.fromYMD(PRIOR_START_YEAR, 1, 1)
    .millis()
)

prior_end_ms = (
    ee.Date.fromYMD(PRIOR_END_YEAR + 1, 1, 1)
    .millis()
)

reburn_start_ms = (
    ee.Date.fromYMD(REBURN_START_YEAR, 1, 1)
    .millis()
)

reburn_end_ms = (
    ee.Date.fromYMD(REBURN_END_YEAR + 1, 1, 1)
    .millis()
)


# Fires intersecting Highline during each screening window
prior_fires = (
    mtbs_boundaries
    .filterBounds(highline_geom)
    .filter(
        ee.Filter.gte(
            "Ig_Date",
            prior_start_ms
        )
    )
    .filter(
        ee.Filter.lt(
            "Ig_Date",
            prior_end_ms
        )
    )
)

reburn_fires = (
    mtbs_boundaries
    .filterBounds(highline_geom)
    .filter(
        ee.Filter.gte(
            "Ig_Date",
            reburn_start_ms
        )
    )
    .filter(
        ee.Filter.lt(
            "Ig_Date",
            reburn_end_ms
        )
    )
)


# Convert qualifying fire polygons into binary raster masks
prior_fire_mask = (
    ee.Image()
    .byte()
    .paint(prior_fires, 1)
    .rename("prior_fire")
)

reburn_mask = (
    ee.Image()
    .byte()
    .paint(reburn_fires, 1)
    .rename("reburn")
)


print(
    "Prior-fire events intersecting Highline:",
    prior_fires.size().getInfo()
)

print(
    "Reburn-period events intersecting Highline:",
    reburn_fires.size().getInfo()
)

Prior-fire events intersecting Highline: 3
Reburn-period events intersecting Highline: 2


#### Step 4A.2 observations

MTBS identified:

- 3 wildfire events intersecting the Highline perimeter during the provisional 2010–2016 prior-fire window;
- 2 wildfire events intersecting the perimeter during the 2018–2022 reburn window.

Event counts alone do not determine patch exclusion because individual fires may overlap only small portions of the focal-fire area. The next step measures the actual overlap fraction for each 90 m modeling patch.

In [19]:
# Step 4A.3 — Summarize MTBS disturbance overlap
# onto the fixed 90 m patches

pixel_area = ee.Image.pixelArea()

prior_area = (
    pixel_area
    .updateMask(prior_fire_mask.eq(1))
    .rename("prior_fire_area_m2")
)

reburn_area = (
    pixel_area
    .updateMask(reburn_mask.eq(1))
    .rename("reburn_area_m2")
)

disturbance_area_image = ee.Image.cat([
    prior_area,
    reburn_area,
])


patches_mtbs_disturbance_raw = (
    disturbance_area_image
    .reduceRegions(
        collection=highline_patches,
        reducer=ee.Reducer.sum(),
        scale=30,
        crs=GRID_CRS,
        tileScale=4,
    )
)


def add_mtbs_disturbance_features(feature):

    patch_area = feature.geometry().area(maxError=1)

    prior_area_m2 = safe_number(
        feature,
        "prior_fire_area_m2",
    )

    reburn_area_m2 = safe_number(
        feature,
        "reburn_area_m2",
    )

    prior_fraction = (
        prior_area_m2
        .divide(patch_area)
        .min(1)
    )

    reburn_fraction = (
        reburn_area_m2
        .divide(patch_area)
        .min(1)
    )

    return feature.set({
        "prior_fire_fraction": prior_fraction,
        "reburn_fraction": reburn_fraction,
        "prior_fire_any": prior_fraction.gt(0),
        "reburn_any": reburn_fraction.gt(0),
    })


patches_mtbs_disturbance = (
    patches_mtbs_disturbance_raw
    .map(add_mtbs_disturbance_features)
)


# One compact server request rather than several repeated getInfo() calls.
mtbs_disturbance_qa = ee.Dictionary({
    "patch_rows":
        patches_mtbs_disturbance.size(),

    "prior_fire_mean_fraction":
        patches_mtbs_disturbance
        .aggregate_mean("prior_fire_fraction"),

    "reburn_mean_fraction":
        patches_mtbs_disturbance
        .aggregate_mean("reburn_fraction"),

    "patches_with_prior_fire":
        patches_mtbs_disturbance
        .filter(ee.Filter.eq("prior_fire_any", 1))
        .size(),

    "patches_with_reburn":
        patches_mtbs_disturbance
        .filter(ee.Filter.eq("reburn_any", 1))
        .size(),
}).getInfo()


print("MTBS DISTURBANCE PATCH QA")

for key, value in mtbs_disturbance_qa.items():
    print(key, ":", value)

MTBS DISTURBANCE PATCH QA
patch_rows : 45505
patches_with_prior_fire : 506
patches_with_reburn : 62
prior_fire_mean_fraction : 0.009272989913034263
reburn_mean_fraction : 0.0010553832112300413


#### Step 4A.3 observations

MTBS disturbance history was successfully summarized onto all 45,505 Highline modeling patches.

Results:

- 506 patches have some overlap with a wildfire in the provisional 2010–2016 prior-fire window;
- 62 patches have some overlap with a wildfire during the 2018–2022 reburn window;
- mean prior-fire overlap fraction across all retained patches ≈ 0.0093;
- mean reburn overlap fraction across all retained patches ≈ 0.0011.

These results are close to the earlier whole-fire MTBS POC estimates and confirm that recent prior fire and post-focal reburn are relatively uncommon within Highline.

The continuous overlap fractions are retained for later eligibility decisions. The current `*_any` flags indicate any nonzero spatial overlap and should not by themselves be interpreted as the final exclusion threshold.

### Step 4A recap — MTBS disturbance history

MTBS fire-history data was used to construct patch-level screening variables for disturbances separate from the 2017 Highline focal fire.

For each 90 m patch, the integration now records:

- `prior_fire_fraction` — fraction overlapping another wildfire during the provisional seven-year pre-fire window;
- `reburn_fraction` — fraction overlapping another wildfire between the focal fire and the end of Season 5;
- corresponding nonzero-overlap QA flags.

These variables are intended for modeling-population eligibility and data-quality screening rather than as predictors of recovery.

The seven-year prior-fire lookback and the final minimum overlap required for exclusion remain production decisions.

### Step 4B — LCMS competing-disturbance screening

LCMS provides annual 30 m landscape-change classifications that are used as patch-level disturbance QA rather than model predictors.

The LCMS POC separates change into:

- **clear competing disturbances** — discrete non-wildfire events that may invalidate interpretation of recovery;
- **ambiguous `Other Loss`** — detected fast vegetation loss whose cause is not confidently identified.

Wildfire itself is not screened with LCMS because prior fire and reburn are handled more reliably with MTBS.

For the Highline integration:

- 2007–2016 is retained as the broad 10-year pre-fire LCMS audit used in the POC;
- 2017–2022 is audited from the focal-fire calendar year through Year 5.

LCMS annual labels are treated as annual QA signals rather than exact disturbance dates. Final pre-fire lookback and patch-exclusion thresholds remain Phase 3 population-level decisions.

In [ ]:
# Step 4B.1 — Load LCMS and define the POC QA contract

lcms = ee.ImageCollection(
    "projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11"
)

lcms_highline = (
    lcms
    .filter(ee.Filter.eq("study_area", "CONUS"))
    .filterBounds(highline_geom)
)


def get_lcms_year(year):
    return ee.Image(
        lcms_highline
        .filter(ee.Filter.eq("year", year))
        .first()
    )


# Clear competing-disturbance classes established in the LCMS POC.
LCMS_CLEAR_CODES = [
    1,   # Wind
    2,   # Hurricane
    5,   # Inundation
    6,   # Prescribed Fire
    8,   # Mechanical Land Transformation
    9,   # Tree Removal
    10,  # Defoliation
    11,  # Southern Pine Beetle
]

# Ambiguous fast vegetation loss.
LCMS_AMBIGUOUS_CODES = [
    13,  # Other Loss
]


LCMS_PREFIRE_YEARS = list(
    range(2007, 2017)
)

LCMS_RECOVERY_YEARS = list(
    range(2017, 2023)
)


lcms_reference = (
    get_lcms_year(2017)
    .select("Change")
)


print("LCMS INTEGRATION CONTRACT")

print(
    "Available years:",
    ee.Image(lcms_highline.sort("year").first())
    .get("year").getInfo(),
    "through",
    ee.Image(lcms_highline.sort("year", False).first())
    .get("year").getInfo(),
)

print(
    "Nominal scale:",
    lcms_reference
    .projection()
    .nominalScale()
    .getInfo(),
    "m"
)

print(
    "Pre-fire audit:",
    LCMS_PREFIRE_YEARS[0],
    "through",
    LCMS_PREFIRE_YEARS[-1],
)

print(
    "Recovery QA:",
    LCMS_RECOVERY_YEARS[0],
    "through",
    LCMS_RECOVERY_YEARS[-1],
)

print(
    "Clear disturbance codes:",
    LCMS_CLEAR_CODES
)

print(
    "Ambiguous-loss codes:",
    LCMS_AMBIGUOUS_CODES
)

LCMS INTEGRATION CONTRACT
Available years: 1985 through 2025
Nominal scale: 30 m
Pre-fire audit: 2007 through 2016
Recovery QA: 2017 through 2022
Clear disturbance codes: [1, 2, 5, 6, 8, 9, 10, 11]
Ambiguous-loss codes: [13]


#### Step 4B.1 observations

The LCMS integration successfully reproduces the POC data and screening contract.

- LCMS coverage spans 1985–2025.
- The annual `Change` product has a nominal 30 m resolution.
- Highline pre-fire QA uses the broad 2007–2016 POC audit window.
- Recovery-period QA covers 2017–2022.
- Eight LCMS classes are grouped as clear competing disturbances.
- `Other Loss` is retained separately as an ambiguous-loss signal.

Wildfire is excluded from these LCMS disturbance groups because prior-fire and reburn screening are handled separately with MTBS.

In [24]:
# Step 4B.2 — Build LCMS disturbance-window surfaces

def lcms_change_mask(year, codes):
    """Binary mask for selected LCMS Change classes in one year."""

    change = (
        get_lcms_year(year)
        .select("Change")
    )

    return (
        change
        .remap(
            codes,
            [1] * len(codes),
            0,
        )
        .rename("flag")
    )


def any_lcms_change(years, codes):
    """Union: location flagged if affected in ANY year."""

    yearly_masks = [
        lcms_change_mask(year, codes)
        for year in years
    ]

    return (
        ee.ImageCollection.fromImages(yearly_masks)
        .max()
    )


def first_lcms_change_year(years, codes):
    """Earliest year in the window with the selected change."""

    yearly_images = []

    for year in years:

        mask = lcms_change_mask(
            year,
            codes,
        )

        year_image = (
            ee.Image.constant(year)
            .toInt16()
            .updateMask(mask.eq(1))
            .rename("first_year")
        )

        yearly_images.append(year_image)

    return (
        ee.ImageCollection.fromImages(yearly_images)
        .min()
    )


# Pre-fire spatial unions
lcms_prefire_clear = any_lcms_change(
    LCMS_PREFIRE_YEARS,
    LCMS_CLEAR_CODES,
)

lcms_prefire_ambiguous = any_lcms_change(
    LCMS_PREFIRE_YEARS,
    LCMS_AMBIGUOUS_CODES,
)


# Focal-fire year through Year 5 spatial unions
lcms_postfire_clear = any_lcms_change(
    LCMS_RECOVERY_YEARS,
    LCMS_CLEAR_CODES,
)

lcms_postfire_ambiguous = any_lcms_change(
    LCMS_RECOVERY_YEARS,
    LCMS_AMBIGUOUS_CODES,
)


# First detected post-fire year, when applicable
lcms_postfire_clear_year = first_lcms_change_year(
    LCMS_RECOVERY_YEARS,
    LCMS_CLEAR_CODES,
)

lcms_postfire_ambiguous_year = first_lcms_change_year(
    LCMS_RECOVERY_YEARS,
    LCMS_AMBIGUOUS_CODES,
)


print("LCMS WINDOW SURFACES CREATED")
print("Pre-fire clear disturbance: ready")
print("Pre-fire ambiguous loss: ready")
print("Post-fire clear disturbance: ready")
print("Post-fire ambiguous loss: ready")
print("Post-fire first-detection year surfaces: ready")

LCMS WINDOW SURFACES CREATED
Pre-fire clear disturbance: ready
Pre-fire ambiguous loss: ready
Post-fire clear disturbance: ready
Post-fire ambiguous loss: ready
Post-fire first-detection year surfaces: ready


#### Step 4B.2 observations

The LCMS disturbance-window surfaces were constructed successfully.

Four spatial-union masks represent whether each location was affected at least once by:

- a clear competing disturbance before the focal fire;
- ambiguous `Other Loss` before the focal fire;
- a clear competing disturbance from the focal-fire year through Year 5;
- ambiguous `Other Loss` from the focal-fire year through Year 5.

Separate surfaces also retain the first post-fire year in which clear disturbance or ambiguous loss was detected.

Repeated detections at the same location across multiple years are not double-counted.

In [22]:
# Step 4B.3a — LCMS disturbance fractions by 90 m patch

lcms_pixel_area = ee.Image.pixelArea()

lcms_disturbance_area_image = ee.Image.cat([
    lcms_pixel_area
    .updateMask(lcms_prefire_clear.eq(1))
    .rename("prefire_clear_area_m2"),

    lcms_pixel_area
    .updateMask(lcms_prefire_ambiguous.eq(1))
    .rename("prefire_ambiguous_area_m2"),

    lcms_pixel_area
    .updateMask(lcms_postfire_clear.eq(1))
    .rename("postfire_clear_area_m2"),

    lcms_pixel_area
    .updateMask(lcms_postfire_ambiguous.eq(1))
    .rename("postfire_ambiguous_area_m2"),
])


patches_lcms_area_raw = (
    lcms_disturbance_area_image
    .reduceRegions(
        collection=highline_patches,
        reducer=ee.Reducer.sum(),
        scale=30,
        tileScale=4,
    )
)


def add_lcms_fraction_features(feature):

    patch_area = feature.geometry().area(maxError=1)

    pre_clear = safe_number(
        feature,
        "prefire_clear_area_m2",
    )

    pre_ambiguous = safe_number(
        feature,
        "prefire_ambiguous_area_m2",
    )

    post_clear = safe_number(
        feature,
        "postfire_clear_area_m2",
    )

    post_ambiguous = safe_number(
        feature,
        "postfire_ambiguous_area_m2",
    )

    return feature.set({
        "prefire_clear_disturbance_fraction":
            pre_clear.divide(patch_area).min(1),

        "prefire_ambiguous_loss_fraction":
            pre_ambiguous.divide(patch_area).min(1),

        "postfire_clear_disturbance_fraction":
            post_clear.divide(patch_area).min(1),

        "postfire_ambiguous_loss_fraction":
            post_ambiguous.divide(patch_area).min(1),
    })


patches_lcms_fractions = (
    patches_lcms_area_raw
    .map(add_lcms_fraction_features)
)


lcms_fraction_qa = ee.Dictionary({

    "patch_rows":
        patches_lcms_fractions.size(),

    "prefire_clear_mean":
        patches_lcms_fractions.aggregate_mean(
            "prefire_clear_disturbance_fraction"
        ),

    "prefire_ambiguous_mean":
        patches_lcms_fractions.aggregate_mean(
            "prefire_ambiguous_loss_fraction"
        ),

    "postfire_clear_mean":
        patches_lcms_fractions.aggregate_mean(
            "postfire_clear_disturbance_fraction"
        ),

    "postfire_ambiguous_mean":
        patches_lcms_fractions.aggregate_mean(
            "postfire_ambiguous_loss_fraction"
        ),

}).getInfo()


print("LCMS PATCH-FRACTION QA")

for key, value in lcms_fraction_qa.items():
    print(key, ":", value)

LCMS PATCH-FRACTION QA
patch_rows : 45505
postfire_ambiguous_mean : 0.01318958440406534
postfire_clear_mean : 5.1568301231657606e-06
prefire_ambiguous_mean : 0.0023062539121636635
prefire_clear_mean : 6.670596032438744e-07


#### Step 4B.3a observations

LCMS disturbance fractions were successfully summarized onto all 45,505 Highline modeling patches.

Mean patch-level fractions are:

- pre-fire clear competing disturbance ≈ 0.0000007 (0.00007%);
- pre-fire ambiguous `Other Loss` ≈ 0.00231 (0.231%);
- post-fire clear competing disturbance ≈ 0.0000052 (0.00052%);
- post-fire ambiguous `Other Loss` ≈ 0.01319 (1.319%).

The ambiguous-loss fractions closely reproduce the earlier whole-fire LCMS POC results. Clear competing disturbances remain effectively absent across Highline; small differences from the already tiny whole-fire percentages are expected when summarizing the retained 90 m patch population instead of the exact irregular perimeter.

In [25]:
# Step 4B.3b — First post-fire LCMS detection year

lcms_timing_image = ee.Image.cat([
    lcms_postfire_clear_year
    .rename("postfire_clear_disturbance_year"),

    lcms_postfire_ambiguous_year
    .rename("postfire_ambiguous_loss_year"),
])


patches_lcms_timing = (
    lcms_timing_image
    .reduceRegions(
        collection=highline_patches,
        reducer=ee.Reducer.min(),
        scale=30,
        tileScale=4,
    )
)


lcms_timing_qa = ee.Dictionary({
    "patch_rows":
        patches_lcms_timing.size(),

    "patches_with_clear_year":
        patches_lcms_timing
        .filter(
            ee.Filter.notNull([
                "postfire_clear_disturbance_year"
            ])
        )
        .size(),

    "patches_with_ambiguous_year":
        patches_lcms_timing
        .filter(
            ee.Filter.notNull([
                "postfire_ambiguous_loss_year"
            ])
        )
        .size(),

    "earliest_clear_year":
        patches_lcms_timing
        .aggregate_min(
            "postfire_clear_disturbance_year"
        ),

    "earliest_ambiguous_year":
        patches_lcms_timing
        .aggregate_min(
            "postfire_ambiguous_loss_year"
        ),
}).getInfo()


print("LCMS TIMING QA")

for key, value in lcms_timing_qa.items():
    print(key, ":", value)

LCMS TIMING QA
earliest_ambiguous_year : 2019
earliest_clear_year : 2022
patch_rows : 45505
patches_with_ambiguous_year : 3175
patches_with_clear_year : 2


#### Step 4B.3b observations

LCMS post-fire disturbance timing was successfully summarized for all 45,505 modeling patches.

- 3,175 patches contain some ambiguous `Other Loss` during the focal-fire-to-Year-5 window.
- Only 2 patches contain a clear competing-disturbance classification.
- The earliest ambiguous-loss detection occurs in 2019.
- The earliest clear competing-disturbance detection occurs in 2022.

The relatively large number of patches with ambiguous loss is compatible with the small mean affected fraction because many patches contain only limited disturbed area.

The 2022 timing of the rare clear disturbance also illustrates the LCMS annual-timing limitation: a Year-5 classification cannot by itself establish whether the disturbance occurred before or after the June–September NBR5 target window.

## Step 4 recap — Disturbance QA

Step 4 established patch-level disturbance-screening metadata from MTBS and LCMS.

### MTBS fire history

MTBS identifies recent wildfire history separately from the focal Highline event.

For each 90 m patch, the integration retains:

- `prior_fire_fraction`
- `reburn_fraction`
- corresponding nonzero-overlap QA flags

Using the provisional seven-year prior-fire window, recent prior wildfire and reburn are uncommon within Highline.

### LCMS competing disturbance

LCMS contributes QA metadata for non-wildfire landscape change rather than model predictors.

For each patch, the integration retains:

- pre-fire clear competing-disturbance fraction;
- pre-fire ambiguous `Other Loss` fraction;
- post-fire clear competing-disturbance fraction;
- post-fire ambiguous `Other Loss` fraction;
- first post-fire clear-disturbance detection year;
- first post-fire ambiguous-loss detection year.

Clear competing disturbances are effectively absent across Highline, while ambiguous `Other Loss` is more common but generally affects only a small fraction of patch area.

No patches are excluded during the Phase 2 POC. Final prior-fire lookback lengths, spatial-overlap thresholds, LCMS exclusion rules, and Year-5 timing treatment remain Phase 3 population-level decisions.

## Step 5 — Final table assembly, validation, and export

### Step 5A — Assemble the integrated patch table

The validated source-specific patch outputs are kept separate until final assembly by `patch_id`. This avoids extending all sources into one unnecessarily deep Earth Engine computation graph.

The Phase 2 table combines:

- focal-fire, vegetation, terrain, and Landsat features;
- Daymet climate features;
- MTBS prior-fire and reburn QA;
- LCMS competing-disturbance QA.

Only the fields required for modeling, interpretation, or downstream QA are retained.

In [45]:
# Step 5A.1 — Define compact source-table schemas

main_fields = [
    # Identity / focal fire
    "patch_id",
    "fire_overlap_fraction",
    "fire_id",
    "fire_name",
    "fire_year",
    "ignition_month",
    "fire_size_acres",

    # MTBS focal severity
    "mtbs_valid_fraction",
    "mtbs_unburned_low_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "mtbs_increased_greenness_fraction",
    "mtbs_nonmapping_fraction",
    "mtbs_meaningful_burn_fraction",

    # LANDFIRE
    "forest_fraction",
    "dominant_evt",
    "dominant_evt_fraction",
    "tree_cover_valid_fraction",
    "tree_cover_mean",
    "tree_height_valid_fraction",
    "tree_height_mean",

    # 3DEP
    "elevation_mean",
    "slope_mean",
    "northness_mean",
    "eastness_mean",

    # Landsat
    "nbr_pre_mean",
    "nbr_1_mean",
    "nbr_5_mean",
    "nbr_drop",
    "nbr_pre_valid_fraction",
    "nbr_1_valid_fraction",
    "nbr_5_valid_fraction",
]

daymet_fields = (
    ["patch_id"]
    + daymet_primary_predictors
)

mtbs_disturbance_fields = [
    "patch_id",
    "prior_fire_fraction",
    "reburn_fraction",
]

lcms_fraction_fields = [
    "patch_id",
    "prefire_clear_disturbance_fraction",
    "prefire_ambiguous_loss_fraction",
    "postfire_clear_disturbance_fraction",
    "postfire_ambiguous_loss_fraction",
]

lcms_timing_fields = [
    "patch_id",
    "postfire_clear_disturbance_year",
    "postfire_ambiguous_loss_year",
]


def compact_table(collection, fields):
    """Keep selected properties and remove geometry."""
    return collection.map(
        lambda feature: ee.Feature(
            None,
            feature.toDictionary(fields)
        )
    )


main_table_ee = compact_table(
    patches_landsat,
    main_fields,
)

daymet_table_ee = compact_table(
    daymet_patch_features,
    daymet_fields,
)

mtbs_disturbance_table_ee = compact_table(
    patches_mtbs_disturbance,
    mtbs_disturbance_fields,
)

lcms_fraction_table_ee = compact_table(
    patches_lcms_fractions,
    lcms_fraction_fields,
)

lcms_timing_table_ee = compact_table(
    patches_lcms_timing,
    lcms_timing_fields,
)


print("FINAL ASSEMBLY SCHEMA")
print("Main/static/Landsat fields:", len(main_fields))
print("Daymet fields:", len(daymet_fields))
print("MTBS disturbance fields:", len(mtbs_disturbance_fields))
print("LCMS fraction fields:", len(lcms_fraction_fields))
print("LCMS timing fields:", len(lcms_timing_fields))
print("\nCompact Earth Engine tables ready.")

FINAL ASSEMBLY SCHEMA
Main/static/Landsat fields: 33
Daymet fields: 17
MTBS disturbance fields: 3
LCMS fraction fields: 5
LCMS timing fields: 3

Compact Earth Engine tables ready.


#### Step 5A.1 observations

Five compact patch-level tables were defined for final assembly:

- 33 main fire/static/Landsat fields;
- 16 Daymet predictors plus `patch_id`;
- 2 MTBS disturbance fields plus `patch_id`;
- 4 LCMS disturbance-fraction fields plus `patch_id`;
- 2 LCMS disturbance-timing fields plus `patch_id`.

All tables use `patch_id` as the common join key and omit geometry from the tabular copies.

In [46]:
# Step 5A.2a — Materialize LCMS timing table locally

def ee_table_to_pandas(collection):
    """Evaluate an Earth Engine FeatureCollection as a Pandas DataFrame."""
    return ee.data.computeFeatures({
        "expression": collection,
        "fileFormat": "PANDAS_DATAFRAME",
    })


lcms_timing_df = ee_table_to_pandas(
    lcms_timing_table_ee
)

print("LCMS TIMING TABLE")
print("Shape:", lcms_timing_df.shape)
print("Columns:", lcms_timing_df.columns.tolist())

display(
    lcms_timing_df.head()
)

LCMS TIMING TABLE
Shape: (45505, 4)
Columns: ['geo', 'patch_id', 'postfire_ambiguous_loss_year', 'postfire_clear_disturbance_year']


,geo,patch_id,postfire_ambiguous_loss_year,postfire_clear_disturbance_year
0,None,HIGHLINE_7212_55640,NaN,NaN
1,None,HIGHLINE_7213_55640,NaN,NaN
2,None,HIGHLINE_7212_55641,NaN,NaN
3,None,HIGHLINE_7213_55641,NaN,NaN
4,None,HIGHLINE_7211_55642,NaN,NaN


#### Step 5A.2a observations

The LCMS disturbance-timing FeatureCollection was successfully materialized locally as a Pandas DataFrame with 45,505 rows.

The two timing fields are preserved by `patch_id`. Missing timing values correctly represent patches with no qualifying LCMS disturbance during the recovery window.

Earth Engine also returned an empty `geo` bookkeeping column; this will be removed consistently during final table cleanup.

In [47]:
# Step 5A.2b — Materialize LCMS disturbance fractions locally

lcms_fraction_df = ee_table_to_pandas(
    lcms_fraction_table_ee
)

print("LCMS FRACTION TABLE")
print("Shape:", lcms_fraction_df.shape)
print("Columns:", lcms_fraction_df.columns.tolist())

display(
    lcms_fraction_df.head()
)

LCMS FRACTION TABLE
Shape: (45505, 6)
Columns: ['geo', 'patch_id', 'postfire_ambiguous_loss_fraction', 'postfire_clear_disturbance_fraction', 'prefire_ambiguous_loss_fraction', 'prefire_clear_disturbance_fraction']


,geo,patch_id,postfire_ambiguous_loss_fraction,postfire_clear_disturbance_fraction,prefire_ambiguous_loss_fraction,prefire_clear_disturbance_fraction
0,None,HIGHLINE_7212_55640,0.0,0.0,0.0,0.0
1,None,HIGHLINE_7213_55640,0.0,0.0,0.0,0.0
2,None,HIGHLINE_7212_55641,0.0,0.0,0.0,0.0
3,None,HIGHLINE_7213_55641,0.0,0.0,0.0,0.0
4,None,HIGHLINE_7211_55642,0.0,0.0,0.0,0.0


#### Step 5A.2b observations

The LCMS disturbance-fraction table was successfully materialized locally with 45,505 patch rows.

Each row retains four patch-level QA fractions:

- pre-fire clear competing disturbance;
- pre-fire ambiguous `Other Loss`;
- post-fire clear competing disturbance;
- post-fire ambiguous `Other Loss`.

Each fraction represents the share of the 90 m patch affected at least once during the corresponding time window, using the spatial union across years rather than double-counting repeated detections.

The first displayed patches contain no qualifying LCMS disturbance and therefore have zero-valued fractions. The empty `geo` column is Earth Engine bookkeeping and will be removed during final table cleanup.

In [48]:
# Step 5A.2c — Materialize MTBS disturbance history locally

mtbs_disturbance_df = ee_table_to_pandas(
    mtbs_disturbance_table_ee
)

print("MTBS DISTURBANCE TABLE")
print("Shape:", mtbs_disturbance_df.shape)
print("Columns:", mtbs_disturbance_df.columns.tolist())

display(
    mtbs_disturbance_df.head()
)

MTBS DISTURBANCE TABLE
Shape: (45505, 4)
Columns: ['geo', 'patch_id', 'prior_fire_fraction', 'reburn_fraction']


,geo,patch_id,prior_fire_fraction,reburn_fraction
0,None,HIGHLINE_7212_55640,0.0,0.0
1,None,HIGHLINE_7213_55640,0.0,0.0
2,None,HIGHLINE_7212_55641,0.0,0.0
3,None,HIGHLINE_7213_55641,0.0,0.0
4,None,HIGHLINE_7211_55642,0.0,0.0


#### Step 5A.2c observations

The MTBS disturbance-history table was successfully materialized locally with 45,505 patch rows.

Each row retains two wildfire-history QA fields:

- `prior_fire_fraction`: fraction of the patch overlapped by another wildfire during the pre-fire lookback window;
- `reburn_fraction`: fraction overlapped by another wildfire after the focal fire through Year 5.

The first displayed patches have no prior-fire or reburn overlap and therefore contain zero-valued fractions.

The empty `geo` column is Earth Engine bookkeeping and will be removed during final table cleanup.

In [49]:
# Step 5A.2d — Materialize Daymet climate features locally

daymet_df = ee_table_to_pandas(
    daymet_table_ee
)

print("DAYMET TABLE")
print("Shape:", daymet_df.shape)
print("Columns:", daymet_df.columns.tolist())

display(
    daymet_df.head()
)

DAYMET TABLE
Shape: (45505, 18)
Columns: ['geo', 'ante_prcp_anom_mm', 'ante_prcp_mm', 'ante_tmax_anom_c', 'ante_tmax_c', 'ante_tmin_anom_c', 'ante_tmin_c', 'ante_vpd_anom_kpa', 'ante_vpd_kpa', 'hist_prcp_mm', 'hist_s1_vpd_kpa', 'hist_tmax_c', 'hist_tmin_c', 'patch_id', 's1_prcp_anom_mm', 's1_tmax_anom_c', 's1_tmin_anom_c', 's1_vpd_anom_kpa']


,geo,ante_prcp_anom_mm,ante_prcp_mm,ante_tmax_anom_c,ante_tmax_c,ante_tmin_anom_c,ante_tmin_c,ante_vpd_anom_kpa,ante_vpd_kpa,hist_prcp_mm,hist_s1_vpd_kpa,hist_tmax_c,hist_tmin_c,patch_id,s1_prcp_anom_mm,s1_tmax_anom_c,s1_tmin_anom_c,s1_vpd_anom_kpa
0,None,113.681002,550.200001,-0.313046,5.308675,1.604074,-5.158146,-0.040261,0.265863,162.254,1.018105,21.429073,4.781771,HIGHLINE_7212_55640,-51.654001,-0.519976,0.769869,-0.03468
1,None,113.681002,550.200001,-0.313046,5.308675,1.604074,-5.158146,-0.040261,0.265863,162.254,1.018105,21.429073,4.781771,HIGHLINE_7213_55640,-51.654001,-0.519976,0.769869,-0.03468
2,None,113.681002,550.200001,-0.313046,5.308675,1.604074,-5.158146,-0.040261,0.265863,162.254,1.018105,21.429073,4.781771,HIGHLINE_7212_55641,-51.654001,-0.519976,0.769869,-0.03468
3,None,113.681002,550.200001,-0.313046,5.308675,1.604074,-5.158146,-0.040261,0.265863,162.254,1.018105,21.429073,4.781771,HIGHLINE_7213_55641,-51.654001,-0.519976,0.769869,-0.03468
4,None,113.681002,550.200001,-0.313046,5.308675,1.604074,-5.158146,-0.040261,0.265863,162.254,1.018105,21.429073,4.781771,HIGHLINE_7211_55642,-51.654001,-0.519976,0.769869,-0.03468


#### Step 5A.2d observations

The Daymet climate table was successfully materialized locally with 45,505 patch rows and all 16 primary climate predictors.

The retained fields represent:

- historical growing-season precipitation, temperature, and VPD;
- antecedent January–May climate before Season 1;
- antecedent anomalies relative to historical conditions;
- Season-1 climate anomalies relative to historical growing-season conditions.

Several neighboring 90 m patches have identical climate values because Daymet is much coarser spatially than the modeling grid. These repeated values reflect the source dataset's native climate resolution rather than artificially implying 90 m climate detail.

The empty `geo` field will be removed during final table cleanup.

In [50]:
# Step 5A.2e — Materialize main integrated feature table locally

main_df = ee_table_to_pandas(
    main_table_ee
)

print("MAIN INTEGRATED TABLE")
print("Shape:", main_df.shape)
print("Columns:", main_df.columns.tolist())

display(
    main_df.head()
)

MAIN INTEGRATED TABLE
Shape: (45505, 34)
Columns: ['geo', 'dominant_evt', 'dominant_evt_fraction', 'eastness_mean', 'elevation_mean', 'fire_id', 'fire_name', 'fire_overlap_fraction', 'fire_size_acres', 'fire_year', 'forest_fraction', 'ignition_month', 'mtbs_high_fraction', 'mtbs_increased_greenness_fraction', 'mtbs_low_fraction', 'mtbs_meaningful_burn_fraction', 'mtbs_moderate_fraction', 'mtbs_nonmapping_fraction', 'mtbs_unburned_low_fraction', 'mtbs_valid_fraction', 'nbr_1_mean', 'nbr_1_valid_fraction', 'nbr_5_mean', 'nbr_5_valid_fraction', 'nbr_drop', 'nbr_pre_mean', 'nbr_pre_valid_fraction', 'northness_mean', 'patch_id', 'slope_mean', 'tree_cover_mean', 'tree_cover_valid_fraction', 'tree_height_mean', 'tree_height_valid_fraction']


,geo,dominant_evt,dominant_evt_fraction,eastness_mean,elevation_mean,fire_id,fire_name,fire_overlap_fraction,fire_size_acres,fire_year,...,nbr_drop,nbr_pre_mean,nbr_pre_valid_fraction,northness_mean,patch_id,slope_mean,tree_cover_mean,tree_cover_valid_fraction,tree_height_mean,tree_height_valid_fraction
0,None,3235.0,1.000000,0.889243,2002.822426,ID4549011522720170729,HIGHLINE,1.000000,93699,2017,...,0.095990,0.184942,1.000000,-0.421951,HIGHLINE_7212_55640,35.885341,25.334750,1.000000,9.685441,1.000000
1,None,3235.0,0.897546,0.750346,1952.352751,ID4549011522720170729,HIGHLINE,0.975845,93699,2017,...,0.139113,0.219511,0.977820,-0.620436,HIGHLINE_7213_55640,34.745095,32.765679,0.999595,9.547909,0.999595
2,None,3235.0,1.000000,0.823176,2034.807865,ID4549011522720170729,HIGHLINE,1.000000,93699,2017,...,0.171700,0.259439,1.000000,-0.523813,HIGHLINE_7212_55641,33.799955,28.909297,1.000000,9.389616,1.000000
3,None,3235.0,0.943251,0.830189,1986.802625,ID4549011522720170729,HIGHLINE,1.000000,93699,2017,...,0.160396,0.200448,1.000000,-0.488127,HIGHLINE_7213_55641,33.103534,33.111888,0.996126,9.565559,0.996126
4,None,3227.0,0.374010,0.251095,2098.646609,ID4549011522720170729,HIGHLINE,0.993716,93699,2017,...,0.120153,0.260434,0.995726,-0.686445,HIGHLINE_7211_55642,27.063942,33.312263,0.889609,10.000000,0.889609


#### Step 5A.2e observations

The main integrated Earth Engine table was successfully materialized locally with 45,505 patch rows and all 33 selected fire, vegetation, terrain, and Landsat fields.

This table contains the primary patch-level information produced by the main integration chain:

- focal-fire metadata and MTBS severity composition;
- LANDFIRE vegetation and forest structure;
- 3DEP terrain summaries;
- Landsat pre-fire, Season-1, and Year-5 NBR information.

`nbr_5_mean` is retained as the eventual prediction target, while the remaining Landsat, fire, vegetation, and terrain fields support prediction, interpretation, or data-quality screening.

The additional empty `geo` field is Earth Engine bookkeeping and will be removed before final assembly.

In [51]:
# Step 5A.3 — Clean and join all local patch tables

local_tables = {
    "main": main_df,
    "daymet": daymet_df,
    "mtbs_disturbance": mtbs_disturbance_df,
    "lcms_fraction": lcms_fraction_df,
    "lcms_timing": lcms_timing_df,
}


# Remove Earth Engine's empty geometry bookkeeping column.
for name, df in local_tables.items():
    if "geo" in df.columns:
        df.drop(columns="geo", inplace=True)


# Confirm every source table has one row per patch_id.
print("SOURCE TABLE INTEGRITY")

for name, df in local_tables.items():
    print(
        f"{name:<18}",
        "rows =", len(df),
        "| unique patch_ids =",
        df["patch_id"].nunique(),
    )


# Main table is the authoritative base.
integrated_df = main_df.copy()

for other_df in [
    daymet_df,
    mtbs_disturbance_df,
    lcms_fraction_df,
    lcms_timing_df,
]:
    integrated_df = integrated_df.merge(
        other_df,
        on="patch_id",
        how="left",
        validate="one_to_one",
    )


print("\nFINAL JOIN")
print("Shape:", integrated_df.shape)
print(
    "Unique patch IDs:",
    integrated_df["patch_id"].nunique(),
)

display(
    integrated_df.head()
)

SOURCE TABLE INTEGRITY
main               rows = 45505 | unique patch_ids = 45505
daymet             rows = 45505 | unique patch_ids = 45505
mtbs_disturbance   rows = 45505 | unique patch_ids = 45505
lcms_fraction      rows = 45505 | unique patch_ids = 45505
lcms_timing        rows = 45505 | unique patch_ids = 45505

FINAL JOIN
Shape: (45505, 57)
Unique patch IDs: 45505


,dominant_evt,dominant_evt_fraction,eastness_mean,elevation_mean,fire_id,fire_name,fire_overlap_fraction,fire_size_acres,fire_year,forest_fraction,...,s1_tmin_anom_c,s1_vpd_anom_kpa,prior_fire_fraction,reburn_fraction,postfire_ambiguous_loss_fraction,postfire_clear_disturbance_fraction,prefire_ambiguous_loss_fraction,prefire_clear_disturbance_fraction,postfire_ambiguous_loss_year,postfire_clear_disturbance_year
0,3235.0,1.000000,0.889243,2002.822426,ID4549011522720170729,HIGHLINE,1.000000,93699,2017,0.781638,...,0.769869,-0.03468,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,3235.0,0.897546,0.750346,1952.352751,ID4549011522720170729,HIGHLINE,0.975845,93699,2017,1.000000,...,0.769869,-0.03468,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,3235.0,1.000000,0.823176,2034.807865,ID4549011522720170729,HIGHLINE,1.000000,93699,2017,0.936664,...,0.769869,-0.03468,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,3235.0,0.943251,0.830189,1986.802625,ID4549011522720170729,HIGHLINE,1.000000,93699,2017,1.000000,...,0.769869,-0.03468,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,3227.0,0.374010,0.251095,2098.646609,ID4549011522720170729,HIGHLINE,0.993716,93699,2017,0.774863,...,0.769869,-0.03468,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


#### Step 5A.3 observations

All five local source tables contained 45,505 rows and 45,505 unique `patch_id` values.

The tables were successfully joined one-to-one by `patch_id`, preserving all 45,505 Highline modeling patches and producing a single 57-column integrated DataFrame.

The final columns consist of:

- 33 focal-fire, MTBS severity, LANDFIRE, terrain, and Landsat fields;
- 16 Daymet climate predictors;
- 2 MTBS wildfire-history QA fields;
- 4 LCMS disturbance-fraction QA fields;
- 2 LCMS disturbance-timing QA fields.

Missing LCMS timing values are expected for patches with no detected competing disturbance.

At this point, all Phase 2 source outputs have been combined into one local patch-level table.

In [52]:
# Step 5B.1 — Validate patch alignment and missingness

source_tables = {
    "main": main_df,
    "daymet": daymet_df,
    "mtbs_disturbance": mtbs_disturbance_df,
    "lcms_fraction": lcms_fraction_df,
    "lcms_timing": lcms_timing_df,
}

main_ids = set(main_df["patch_id"])

print("PATCH-ID ALIGNMENT")

for name, df in source_tables.items():
    ids = set(df["patch_id"])

    print(
        f"{name:<18}",
        "| missing vs main =", len(main_ids - ids),
        "| extra vs main =", len(ids - main_ids),
    )


print("\nFINAL TABLE STRUCTURE")
print("Rows:", len(integrated_df))
print("Columns:", integrated_df.shape[1])
print(
    "Unique patch IDs:",
    integrated_df["patch_id"].nunique(),
)
print(
    "Duplicate patch IDs:",
    integrated_df["patch_id"].duplicated().sum(),
)
print(
    "Duplicate column names:",
    integrated_df.columns.duplicated().sum(),
)


# Summarize only columns containing missing values.
missing = integrated_df.isna().sum()
missing = missing[missing > 0].sort_values(
    ascending=False
)

missing_summary = (
    missing
    .rename("missing_rows")
    .to_frame()
)

missing_summary["missing_pct"] = (
    100
    * missing_summary["missing_rows"]
    / len(integrated_df)
)

print(
    "\nColumns containing missing values:",
    len(missing_summary),
)

display(missing_summary)

PATCH-ID ALIGNMENT
main               | missing vs main = 0 | extra vs main = 0
daymet             | missing vs main = 0 | extra vs main = 0
mtbs_disturbance   | missing vs main = 0 | extra vs main = 0
lcms_fraction      | missing vs main = 0 | extra vs main = 0
lcms_timing        | missing vs main = 0 | extra vs main = 0

FINAL TABLE STRUCTURE
Rows: 45505
Columns: 57
Unique patch IDs: 45505
Duplicate patch IDs: 0
Duplicate column names: 0

Columns containing missing values: 12


,missing_rows,missing_pct
postfire_clear_disturbance_year,45503,99.995605
postfire_ambiguous_loss_year,42330,93.022745
tree_cover_mean,206,0.452698
dominant_evt,166,0.364795
tree_height_valid_fraction,166,0.364795
tree_cover_valid_fraction,166,0.364795
dominant_evt_fraction,166,0.364795
eastness_mean,163,0.358202
elevation_mean,163,0.358202
slope_mean,163,0.358202


#### Step 5B.1 observations

All source tables contain the exact same 45,505 `patch_id` values: no source is missing patches or contains additional unmatched patches.

The assembled table contains 45,505 rows and 57 columns, with:

- 45,505 unique patch IDs;
- no duplicate patch IDs;
- no duplicate column names.

Missing LCMS timing values are expected structural missingness rather than failed data extraction. Only patches with a detected disturbance can have a first-disturbance year: 2 patches have a clear competing-disturbance year and 3,175 have an ambiguous-loss year.

Other missingness is very limited (~0.35–0.45%) and is concentrated in LANDFIRE vegetation and forest-masked terrain summaries. Final population-level eligibility and coverage thresholds will be determined in Phase 3 rather than removing rows during the Phase 2 POC.

In [53]:
# Step 5B.2 — Validate key ranges and classify final columns

target_field = "nbr_5_mean"

identifier_metadata_fields = [
    "patch_id",
    "fire_id",
    "fire_name",
    "fire_year",
]

qa_fields = [
    "fire_overlap_fraction",

    "mtbs_valid_fraction",
    "mtbs_nonmapping_fraction",

    "tree_cover_valid_fraction",
    "tree_height_valid_fraction",

    "nbr_pre_valid_fraction",
    "nbr_1_valid_fraction",
    "nbr_5_valid_fraction",

    "prior_fire_fraction",
    "reburn_fraction",

    "prefire_clear_disturbance_fraction",
    "prefire_ambiguous_loss_fraction",
    "postfire_clear_disturbance_fraction",
    "postfire_ambiguous_loss_fraction",
    "postfire_clear_disturbance_year",
    "postfire_ambiguous_loss_year",
]

candidate_predictor_fields = [
    # Focal-fire context
    "ignition_month",
    "fire_size_acres",

    # MTBS severity
    "mtbs_unburned_low_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "mtbs_increased_greenness_fraction",
    "mtbs_meaningful_burn_fraction",

    # LANDFIRE
    "forest_fraction",
    "dominant_evt",
    "dominant_evt_fraction",
    "tree_cover_mean",
    "tree_height_mean",

    # 3DEP
    "elevation_mean",
    "slope_mean",
    "northness_mean",
    "eastness_mean",

    # Landsat
    "nbr_pre_mean",
    "nbr_1_mean",
    "nbr_drop",

    # Daymet
    *daymet_primary_predictors,
]


# Confirm every column has an explicit role.
classified_fields = (
    set(identifier_metadata_fields)
    | set(qa_fields)
    | set(candidate_predictor_fields)
    | {target_field}
)

unclassified = sorted(
    set(integrated_df.columns) - classified_fields
)

unexpected_classified = sorted(
    classified_fields - set(integrated_df.columns)
)

print("COLUMN ROLES")
print("Candidate predictors:", len(candidate_predictor_fields))
print("Target:", target_field)
print("Identifier/metadata fields:", len(identifier_metadata_fields))
print("QA fields:", len(qa_fields))
print("Unclassified columns:", unclassified)
print("Classified fields not present:", unexpected_classified)


# Compact range checks for important variable families.
range_fields = [
    "forest_fraction",
    "mtbs_low_fraction",
    "mtbs_moderate_fraction",
    "mtbs_high_fraction",
    "nbr_pre_mean",
    "nbr_1_mean",
    "nbr_5_mean",
    "nbr_drop",
    "elevation_mean",
    "slope_mean",
    "northness_mean",
    "eastness_mean",
    "tree_cover_mean",
    "tree_height_mean",
]

range_summary = integrated_df[
    range_fields
].agg(["min", "max"]).T

display(range_summary)

COLUMN ROLES
Candidate predictors: 36
Target: nbr_5_mean
Identifier/metadata fields: 4
QA fields: 16
Unclassified columns: []
Classified fields not present: []


,min,max
forest_fraction,0.000000,1.000000
mtbs_low_fraction,0.000000,1.000000
mtbs_moderate_fraction,0.000000,1.000000
mtbs_high_fraction,0.000000,1.000000
nbr_pre_mean,-0.233945,0.720767
nbr_1_mean,-0.516018,0.687016
nbr_5_mean,-0.292240,0.720883
nbr_drop,-0.245757,1.080492
elevation_mean,1484.213454,2541.717716
slope_mean,0.000000,44.637804


#### Step 5B.2 observations

Every column in the 57-field integrated table has an explicit downstream role:

- 36 candidate predictor fields;
- 1 target (`nbr_5_mean`);
- 4 identifier/metadata fields;
- 16 QA or eligibility fields.

No columns remain unclassified, and all expected classified fields are present.

Key numerical ranges are also plausible:

- fraction-valued fields remain within [0, 1];
- northness and eastness remain within approximately [-1, 1];
- NBR values remain within their valid physical range;
- elevation, slope, tree cover, and tree height are consistent with plausible Highline conditions.

These are candidate predictor roles rather than permanently locked model inputs; Phase 3 and Phase 4 may refine eligibility rules or remove redundant predictors using population-level evidence.

In [54]:
# Step 5B.3 — Export the integrated Highline Phase 2 table

from pathlib import Path

# Resolve repository root whether notebook is launched
# from the repo root or from notebooks/.
repo_root = (
    Path.cwd()
    if (Path.cwd() / "pyproject.toml").exists()
    else Path.cwd().parent
)

output_dir = repo_root / "data" / "processed"
output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_path = (
    output_dir
    / "highline_phase2_integrated.csv"
)

integrated_df.to_csv(
    output_path,
    index=False,
)

print("PHASE 2 TABLE EXPORTED")
print("Path:", output_path)
print("Rows:", len(integrated_df))
print("Columns:", integrated_df.shape[1])
print(
    "File size (MB):",
    round(
        output_path.stat().st_size / 1_000_000,
        2,
    ),
)

PHASE 2 TABLE EXPORTED
Path: data\processed\highline_phase2_integrated.csv
Rows: 45505
Columns: 57
File size (MB): 34.55


#### Step 5B.3 observations

The completed Phase 2 Highline integration table was exported successfully to `data/processed/highline_phase2_integrated.csv`.

The exported dataset contains:

- 45,505 modeling patches;
- 57 retained fields;
- one row per `patch_id`;
- all currently retained predictor, target, metadata, and QA information.

No Phase 3 eligibility filtering has yet been applied, so the export preserves the complete Phase 2 Highline patch population.

In [55]:
# Step 5C.1 — Verify exported table round-trip

export_check_df = pd.read_csv(
    output_path
)

print("EXPORTED TABLE VERIFICATION")
print("Rows:", len(export_check_df))
print("Columns:", export_check_df.shape[1])
print(
    "Unique patch IDs:",
    export_check_df["patch_id"].nunique(),
)
print(
    "Duplicate patch IDs:",
    export_check_df["patch_id"].duplicated().sum(),
)
print(
    "Columns match in-memory table:",
    export_check_df.columns.tolist()
    == integrated_df.columns.tolist(),
)

assert export_check_df.shape == integrated_df.shape
assert export_check_df["patch_id"].nunique() == 45505
assert not export_check_df["patch_id"].duplicated().any()
assert (
    export_check_df.columns.tolist()
    == integrated_df.columns.tolist()
)

print("\nExport verification passed.")

EXPORTED TABLE VERIFICATION
Rows: 45505
Columns: 57
Unique patch IDs: 45505
Duplicate patch IDs: 0
Columns match in-memory table: True

Export verification passed.


### Step 5 recap — Final table assembly and validation

Step 5 converted the validated source-specific Earth Engine outputs into one local patch-level modeling table.

Five compact source tables were materialized separately:

1. focal-fire, MTBS severity, LANDFIRE, 3DEP, and Landsat;
2. Daymet climate;
3. MTBS prior-fire and reburn QA;
4. LCMS disturbance-fraction QA;
5. LCMS disturbance-timing QA.

Keeping these outputs separate until materialization avoided extending every source into one unnecessarily deep Earth Engine computation graph.

The local tables were joined one-to-one by `patch_id`. All five sources contained the exact same 45,505 Highline patches, producing a final table with 45,505 rows and 57 fields.

Every retained column was assigned a downstream role:

- 36 candidate predictors;
- 1 target (`nbr_5_mean`);
- 4 identifier/metadata fields;
- 16 QA or eligibility fields.

Structural and numerical validation found no duplicate patches or columns, complete patch-ID alignment across sources, and plausible ranges for the major spectral, vegetation, terrain, and fraction-valued variables.

The final Phase 2 table was exported to `data/processed/highline_phase2_integrated.csv` and successfully verified by reading the saved file back from disk.

## Phase 2 integration handoff

The Highline POC now demonstrates the complete integration architecture required to construct one modeling row per 90 m wildfire patch from all six project data sources.

Phase 2 establishes the source contracts, temporal definitions, spatial summarization methods, feature calculations, QA fields, and final table structure.

The Phase 2 export intentionally retains the full Highline patch population. Population-level decisions remain deferred to Phase 3, including:

- final forest/woodland eligibility thresholds;
- meaningful focal-fire severity requirements;
- prior-fire lookback choice and exclusion threshold;
- reburn exclusion rules;
- LCMS competing-disturbance thresholds and Year-5 timing treatment;
- minimum Landsat and vegetation-data coverage requirements;
- production handling across fires, years, states, and source versions.

Phase 3 will generalize the validated Highline logic into a reproducible multi-fire data pipeline and use population-level evidence to finalize these eligibility rules.